<a href="https://colab.research.google.com/github/hernandezb3/llm-text-classification/blob/main/Private_BINARY__llm_txt_classification_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATASET Persuade Corpus 2.0

https://www.kaggle.com/datasets/nbroad/persaude-corpus-2
* The PERSUADE 2.0 corpus builds on the PERSUADE 1.0 corpus by providing holistic essay scores to each persuasive essay in the PERSUADE 1.0 corpus as well as proficiency scores for each argumentative and discourse element found in the initial corpus. This version also contains all essays (as compared to 1.0 which linked the training set for the Kaggle competition)

* In total, the PERSUADE 2.0 corpus comprises over 25,000 argumentative essays produced by 6th-12th grade students in the United States for 15 prompts on two writing tasks: independent and source-based writing. The PERSUADE 2.0 corpus provides detailed individual and demographic information for each writer as well as the initial annotations for argumentative and discourse element found PERSUADE 1.0.

Packages
* We are going to use a genAI model as a classifier via prompting then evaluate it against gold human labels
* Aumodelforcausallm

## Environment SetUp
### Load 

In [1]:
ENV = "colab" # either local or colab
USER = "brittney"

In [4]:
import os
import re

if ENV == "local":
    !source .venv/bin/activate
    !pip3 install -r requirements.txt
elif ENV == "colab":
    !pip install -r https://raw.githubusercontent.com/hernandezb3/llm-text-classification/main/requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [5]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from huggingface_hub import login

### Load tokens

In [ ]:
if ENV == "local":
    from dotenv import load_dotenv
    load_dotenv()
elif ENV == "colab":
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("B-Llama-3.1-8B-Access")

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN not set — check your .env file or Colab secret.")
elif os.environ.get("HF_TOKEN") == "":
    raise RuntimeError("HF_TOKEN not set — check your .env file or Colab secret.")

HF_TOKEN = os.environ.get("HF_TOKEN")

Kaggle
* PERSUADE 2.0 is a public dataset, and kagglehub can download public datasets anonymously no credentials required.
* kagglehub.dataset_download(...) pulls the dataset from Kaggle and caches it locally, returning the path to that cached copy
*  We are loading one specific file the persuade_2.0_human_scores_demo_id_github.csv into a dataframe

In [ ]:
#load data
import kagglehub

# Download dataset
path = kagglehub.dataset_download("nbroad/persaude-corpus-2")
print("Dataset path:", path)
print("Files:", os.listdir(path))

scores_path = os.path.join(path, "persuade_2.0_human_scores_demo_id_github.csv")
scores = pd.read_csv(scores_path)

print(f"Shape  : {scores.shape}")
print(f"Columns: {scores.columns.tolist()}")
scores.head()

In [ ]:
os.listdir(path)


## Assign Prompt IDS

In [ ]:
# how many prompt topics
prompt_summary = (
    scores.groupby("prompt_name")["holistic_essay_score"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
prompt_summary["prompt_id"] = ["Prompt " + str(i + 1) for i in range(len(prompt_summary))]

prompt_id_map = prompt_summary.set_index("prompt_name")["prompt_id"].to_dict()
scores["prompt_id"] = scores["prompt_name"].map(prompt_id_map)

print(prompt_summary[["prompt_id", "prompt_name", "holistic_essay_score"]]
      .rename(columns={"holistic_essay_score": "mean_score"})
      .round(2)
      .to_string(index=False))

## Binary Classification

In [ ]:
# Scores 1 to 3 = Fail (0), Scores 4 to 6 = Pass (1)
PASS_THRESHOLD = 4

def to_binary(score):
    return 1 if score >= PASS_THRESHOLD else 0

In [ ]:
# total sample is 25,000 too many essay
# lets take 340 from each prompt type. total = 5,100
sample = (
    pd.concat([
        grp.sample(n=min(340, len(grp)), random_state=42)
        for _, grp in scores.groupby("prompt_name", sort=False)
    ])
    .reset_index(drop=True)
)
sample["binary_score"] = sample["holistic_essay_score"].apply(to_binary)


print(f"Total essays  : {len(sample):,}")
print(f"Unique prompts: {sample['prompt_name'].nunique()}")
print()
print(sample["binary_score"].value_counts().rename({0: "Fail", 1: "Pass"}))

Data Split
*  data split
   * Train is the data the model learns from. Its parameters (weights) are fit to this set. This is for finetuning
   * Validation is a held-out set that the model don't train on, it is used during development to tune things such as learning rate, number of epochs, which model, prompt wording, decision thresholds. It is for adjustment
   * Test is a held-out set, it is only touched once, at the very end, to get an honest estimate of how the model does on data it has never influenced in any way.
* Finetuning overfitting? the thing that causes overfitting is too many epochs on too little data


Sample Size
* Training set if for finetuning... 3000 essays
* not sure sample size here, is 3000 too much for finetuning...

In [ ]:
X = sample.drop(columns=["holistic_essay_score", "binary_score"])
y = sample["binary_score"].copy()          # binary 0/1 instead of 1 to 6, th 1 to 6 was not working at all

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train):,}  ({len(X_train)/len(sample)*100:.1f}%)")
print(f"Val   : {len(X_val):,}   ({len(X_val)/len(sample)*100:.1f}%)")
print(f"Test  : {len(X_test):,}   ({len(X_test)/len(sample)*100:.1f}%)")
print("\nClass distribution (stratification check):")
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = y_split.value_counts().rename({0: "Fail", 1: "Pass"})
    print(f"  {name}: {counts.to_dict()}")


In [ ]:
## Sample 340

## MODEL INPUT

**Building the Model Prompt**
* generate_prompt: this is to build the prompt and give it to the llm, this include the
    * Instructions
    * Student essay text
    *  essay label of pass/fail

* generate_test_pumpt: here is similar as above but the prompt stops before providing the label, nolabel provided here

train_df
* Here train df gets the generated prmpt with inlcude the full examples with the labels. Now this is the training data that we will feed to a finetuner LoRA as training needs the answer

test_df
* Here test df gets teh generate test prompt as labels are not given. The model nees to predict the essay pass/failing and such predictions will be evaluated against teh y_true (human evaluated gold standard)

In [ ]:
#Prompt generation

def grading_instruction_prompt(row):
    return (
        f"You are an expert essay grader. Read the essay and respond with exactly one word.\n"
        f"Your response must be either the word Fail or Pass. No other words.\n\n"
        f"Rules:\n"
        f"- Fail: the essay is weak, underdeveloped, or below standard\n"
        f"- Pass: the essay is proficient, strong, or excellent\n\n"
        f"Prompt: {row['prompt_name']}\n"
        f"Task: {row['task']}\n"
        f"Essay: {row['full_text']}\n\n"
        f"Respond with one word only (Fail or Pass): "
    )


def make_prompt_completion(row, tokenizer):
    user_content = grading_instruction_prompt(row)   # same wording
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False, add_generation_prompt=True,
    )
    completion = "Pass" if row["binary_score"] == 1 else "Fail"
    return {"prompt": prompt, "completion": completion}

#Train
train_df = X_train.copy()
train_df["binary_score"] = y_train.values

# eval prompt
val_df = X_val.copy()
val_df["binary_score"] = y_val.values
X_val_prompts = pd.DataFrame(val_df.apply(grading_instruction_prompt, axis=1), columns=["text"])

test_df = X_test.copy()
test_df["binary_score"] = y_test.values
y_true = test_df["binary_score"].values
X_test_prompts = pd.DataFrame(test_df.apply(grading_instruction_prompt, axis=1), columns=["text"])

val_df
* X_val_prompt validation data that does not have the labels
* val_df have the labels it got it from  generate prompt

In [ ]:
print(f"Train      : {len(train_df)}")
print(f"Val prompts: {len(X_val_prompts)}")
print(f"Test prompts: {len(X_test_prompts)}")

In [ ]:
#balanced for training
fail_df = train_df[train_df["binary_score"] == 0]
pass_df = train_df[train_df["binary_score"] == 1]

train_balanced = pd.concat([
    fail_df.sample(n=len(pass_df), random_state=42),
    pass_df,
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced: {train_balanced['binary_score'].value_counts().to_dict()}")

In [ ]:
def predict_decoder_only(test, model, tokenizer, verbose=False):
    """Zero-shot or finetuned inference for decoder-only instruct model. So this can
    served tinyllama, gemma, mistral, phi-3.5, qwen2 and llama-3.1 unchanged.

    Why? universal because apply_chat_template emits each model's own markers
    (<|assistant|>, [INST], ChatML, ...)
    The slice exists because decoder-only output = prompt + answer in one stream.

    Returns (y_pred, y_generated): 1=Pass, 0=Fail, -1=unparseable.
    Many -1s = the format/extraction is wrong, not the model."""
    y_pred      = []
    y_generated = []

    model.eval()   # disable training-only behavior (dropout)

    # pad token dedicated pad if the model has one,
    # else EOS (first element if EOS is a list, e.g. Gemma's [1, 107])
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        eos = tokenizer.eos_token_id
        pad_id = eos[0] if isinstance(eos, list) else eos

    for i in tqdm(range(len(test))):

        # wrap the prompt in this model's chat format
        messages = [{"role": "user", "content": test.iloc[i]["text"]}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096,   # OOM guard, not a constraint...prompts run ~700-900 tokens,
                               # (Real ceiling per model:
                               # look into model.config.max_position_embeddings.)
            padding=False,     # one essay at a time, nothing to pad against right now
        ).to(model.device)

        # decoder-only models return input + answer together
        # remember the input length so we can cut the prompt back off
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,     # we only need one word ("Pass"/"Fail")
                do_sample=False,       # greedy = deterministic = reproducible
                pad_token_id=pad_id,
                use_cache=False,       # kept off since the Phi remote-code cache bug errors
                                       # harmless for native classes at 10 new tokens
            )

        # keep only what the model added, after the prompt
        new_tokens = outputs[0][input_len:]
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        y_generated.append(generated)

        # parse: contains "pass" -> 1, "fail" -> 0, neither -> -1 (unparseable)
        if   "pass" in generated: y_pred.append(1)
        elif "fail" in generated: y_pred.append(0)
        else:                     y_pred.append(-1)

        if verbose:
            print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {y_pred[-1]}")

    return y_pred, y_generated

# THE MODELS INFO

### **Tokenizer info**

AutoTokenizer loads the tokenizer that matches a model
1. Text ↔ token IDs: As we know models dont see words. They see ID numbers from fixed vocabulary

AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
* The tokenizer is the model's dictionary (Language)
* downloads (or reads) the tokenizer files from that Hub repo related to that specific model (the vocabulary and merge rules)
* tokenizer_config.json (settings, special tokens, and the chat template if the model has one).


The tokenizer and model are inseparable partners
* The model's embedding table has exactly one row per vocabulary entry. Encode with the wrong tokenizer and ID 8241 points at a different word than the model thinks (complete chaos)

### **Padding info**
What is padding? T-T

I guess it is about filling extra space like placeholders tokens because we need that every sequence in a batch ends up with the same length
 * This is for processing GPU process barches as rectangular grids of numbers can not handle rows with different lengths in a single batch, like matrices calculations

GPU needs every sequence of token lenght to be the same lenght. But lets say essays are of different lenghts.
 * Essay A: "I think cars ar bad."
    * 6 tokens

 * Essay B: "Online school is great."
    * 5 tokens

To Batch, the model will need to pad to length 6
(so we need to padding, this is what i understands)
- Essay A: [i] [think] [cars] [are] [bad] [.]

- Essay B: [online] [school] [is] [great] [.] [PAD]

So now that these are all the same length, they can be stacked into one tensor the GPU can process in parallel. Also, i need to remember that pad has no meeaning when we mask it looks something lik
- Essay 1 input: [I] [like] [cars] [.] [PAD] [PAD]

- Essay 1 input: 1 1 1 1 0 0

(The masking allow the model to ignore the padded positions when computing attention and loss)

### **Model Architecture**
There are three classic families of transformer architecture
1. Encoder only (BERT, RoBERT, DeBERTa)
   * It ingests/reads the whole input at once and builds a representation of it, but it doesn't generate text. Like embeddings, in another project I feed essays to a BERT model and it returned the essays embedding (this was a BERTembedding model)
   * bidirectional attention layers, every tokens can see each other, left and right
2. Decoder only (GPT, Llama, Mistral, TinyLlama, Qwen, Phi)
   * Read input and continues the text, so output = prompt + answer
   * causal left attention, each token see only the token before it, that why instructions, prompt and answer are like glued together, what we are calling the answer/label token is the later position in the left to right sequence
3. Encoder/decoder seq2seq (T5, BART)
   * Encoder/decoder is for transformation tasks such as translations, summarization, input and output are different objects.
   * bidirectional attention layers. Fully reads read input, the a casual layer decoder generates the output.

BUT wait there is more...
* These architecture are different from the model specified training such as...
   * Based Model???: a base model is trained on predicting the text token across a huge pile of text (usually internet). Good at continuation
   * Instruct models???: Takes the based models and these get an extra round of training of millions of dialogue, instructions-->good response
      * like dialogue pair, here each model...wait for it...develop their own MARKER token....so what is a Marker tokennnnnnnnnnnnnn (see more below of course)
      * So, instruct models specify on a dialogue behavior. They starts as base model given additional training to follow instructions and respond as a helpful assistant, rather than just continue text
        * when they see a instruction in the EXPECTED FORMAT it recognizes the instruction and produce a response

Why do we have to know all of the above??
* because of errors..............

### **Code explanation**


**Model Tasks**
* .from_pretrained() method, we use it to load the model weighits
* AutoModelFor...class name to get the import. Now, which model weight??????
    * AutoModelForCausalLM.from_pretrained()
         * decoder only model = output prompt + continuation
   * AutoModelForSequenceClassification()
     * encoder only, the outputs is the label

  * AutoModelForSeq2SeqLM()
    * encoder/decoder, translation, summary and so on
      * outputs the generated text only

**Pad and eos**
  * **tokenizer.eos_token_id **refers to the End Of Sequence token. Its job is to signal "the text is finished." The model emits it to say "I'm done"
    * Do no confuse this with marker tokens, marker token refers to th structural tokens that labels who is talking. This is about Who is talking
      * <|user|>, <|assistant|>, [INST], <start_of_turn>user, <start_of_turn>model
      * OES is different type of tokens whose jobs is to signal "the text is finished, like saying im done

  * **tokenizer.pad_token_id** refers to the padding token. Its job is to fill empty space so multiple sequences of different lengths can be stacked into one rectangular tensor for batching. We are not batching but this is very important

So,
* tokenizer.pad_token = tokenizer.eos_token
   * set the pad and token to be the same value
   * some models doesnt come with a pad token, but they have to come with eos
    * in this case it doesnt matter which we passed to the generate()
    * generate() pad_token_id argument wants a single integer
EOS marks the end of real content, PAD is fake content added to line things up.

**Model Computation**

* Model computation are done using floating-point numbers
The precision of these number impact how accurately the model performs in different task such as training and inferences.
* Floating Point Representation: Mantissa, Exponent, and Sign. A floating-point number is made up of three parts:     
    * The mantissa
      * Mantissa: fraction significant digits more bits allocated here more accurate the number
    * The exponent
      * Exponent the scale or range of the number, how far left or right it should be
    * The sign bit
      * the number sign, a whole bit just dedicated to know if the number is positive or negative, idk why this is important

**dtype nightmares**
* So torch dtype is the numeric precision of the models weights. This is just one line of code but i do argue is the most or one of the most important part to understands.
  * **FP32 (32-bit floating point):** This is the standard data type for many machine learning applications. It provides high precision, with 23 bits for the mantissa and 8 as exponent. However, the high precision comes at the cost of large memory usage and slower computations.
    * PLEASE know we can do float 32 because this is a very very small model
    
  * **FP16 (16-bit Floating Point)**: This is a compromise between speed and accuracy. It reduces memory usage and speeds up computations but sacrifices some precision.
    * For bigger models I use this OR wait for it....yess quantization...

  * **Bfloat16 (Brain Floating Point)**: This is a more recent development designed for deep learning tasks. It has 8 exponent bits but 7 mantissa bits. It keeps the exponent size from FP32 but reduces the mantissa size. Bfloat16 maintains the dynamic range of FP32, making it  useful for training large models, though it still comes with the trade-off of needing specialized hardware and being more expensive than lower-precision types like FP16.
    * ERROR: cannot use bf15 with T4 or even L4 GPU, to use bf16 the newer gpu are needed like A100... (dont want to talk about gpu....)
  

### **Tinyllama model 1.1B**

In [ ]:
# This is model loading
# LOCAL: TinyLlama 1.1B, CPU, PLEASE know that there is NO quantization happening here right now
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #not working all pssed all faild
#based_model = "google/flan-t5-base" #250m maybe less biased

#convert text into numbers, tokenIDs that the model understands
#every model have their own matching tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model) #this is our tokenizer

#this is about padding [PAD]
#each model has a dedicated PAD only and  <EOS> end of sequence token!! (separate)
tokenizer.pad_token = tokenizer.eos_token

#where the padding goes
# to the left for inferences
tokenizer.padding_side = "left"

#auto.. to call a causal language model a model that predicts th next token
#from_pretrained, downloads the pretrained weights for th model
model = AutoModelForCausalLM.from_pretrained(   # remember this is from huggingface to call the model
    base_model,                      # the name of the model
    torch_dtype=torch.float32,       # float32 for CPU stability, T-T every important
                                     # we are controlling hte precision of the loaded weights
                                     # float 16 halves memory usage compared to float32
                                     # GPU T4 does not support bfloat16
    device_map="auto",               # auto tells hf to decide on the avaliable resources: in our case: change "cpu" to "auto" so T4 GPU is used
)

#model and tokenizer aggreement, they need to speack the same language
model.config.pad_token_id = tokenizer.eos_token_id #


In [ ]:
# helpful info if you want to complicate your life more
print(model)
# what element i would lookhere the attention layers
# the max features here is 2048, this is important to know see below


In [ ]:
print(model.config)
#here i will check dtype if quant worked it showed here

In [ ]:
print(model.get_memory_footprint() / 1e9, "GB")
#model size

In [ ]:
print("Model loaded on:", next(model.parameters()).device)
print("Vocab size     :", len(tokenizer))
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

# Sanity check: confirm pass/fail tokens exist in vocabular
pass_id = tokenizer.encode("Pass", add_special_tokens=False)
fail_id = tokenizer.encode("Fail", add_special_tokens=False)
print(f"\nToken check:")
print(f"  'Pass'  token id(s): {pass_id}")
print(f"  'Fail'  token id(s): {fail_id}")

## Chat Format
* No idea how to call this section

### **Models Marker Tokens**
* Models need their specified FORMAT, if you want to talk to them of course. So, if they want you to call them Sir thats how you need to address them. This is all related to MARKER tokens. Lets put more seasoning...

**LLM text-Completer**
at the lowest level, every causal LM (Phi-2, Mistral, Llama, GPT) does exactly something like given some text, predict the next token
   * If we as the user feed a model this

```
The capital of DR is
```
* It continues "Santo Domingo" not because it is answering but because "Santo Domingo" is the statistically most likely continuation

SPECIAL MARKERS TOKENS

**LLM:Instruct Models** like Mistral-7B Instruct, Llama-3-Instruct and every chat assistant starts as a base model, then it get extra round of training on thousands of formatted dialogues (repetition is good for learning, read that again)

```
user message-->assistant response, user message-->assistant response
```
BUT the models see a flat stream of tokens , it/they? doesnt know where the user turn ends and its own begins. So, we need **"Special Marker Tokens"** inserted around each turn during that training. Different models have different markers.

    * Mistral's markers tokens
       - <s>[INST] Classify this essay as Pass or Fail. Essay: ... [/INST]
       - During instruct-training, the model saw millions of examples where the text after [/INST] was a helpful assistant response. So it learned the association: "when I see [/INST], what follows is my answer"

    * llama 3 Instruct
      - <|begin_of_text|><|start_header_id|>user<|end_header_id|>Grade this essay. Essay: ...<|eot_id|><|start_header_id|>assistant<|end_header_id|>Pass<|eot_id|>


    * ChatML (use by Qwen)
     - <|im_start|>user
       Grade this essay. Essay: ...<|im_end|>
       <|im_start|>assistant
       Pass<|im_end|>

So
1. Feed an instruct model its expected format, and it behaves like an assistant.
2. But know that special marker tokens are arbitraty and every model has different one.
   * Use the wrong format on the wrong model and things degrade quietly: the model half-recognizes the structure, output quality drops, and nothing errors out to tell whyyyyyyyyyyyyyyyy.

So,

Do we need to memorize every format (T-T, crying face), the answer is no (:D). Hugging Face stores each model's format inside its tokenizer as a template that we can use for most models (a Jinja string in tokenizer_config.json). We have to write the conversation in a universal shape:
```
messages = [{"role": "user", "content": "Grade this essay. Essay: ..."}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# add_generation_prompt=True appends the assistant-turn opener,
# so the model's next tokens ARE the assistant's reply
```

Messages is in the standard format, then, **apply_chat_template** renders it into that model's markers
  * The add_generation_prompt=True flag appends the "your turn now" model marker
  * if a model does not have have a chat template it will only have \nAnswer:

So, (my last So)
 "role": "user" labels who is speaking in that turn of the conversation, while "content" is the message itself.

 A conversation is a list of these dicts


```
messages = [
    {"role": "system",    "content": "You are a strict essay grader."},   # setup/instructions
    {"role": "user",      "content": "Grade this essay: ..."},            # the human
    {"role": "assistant", "content": "Pass"},                             # the model
    {"role": "user",      "content": "Are you sure? It's quite short."},  # human again
]
```

The main three roles:
* User is the human's turns (requests, questions)
* Assistant is the model's turns (its previous answers), the model needs to identify its own answer
* System is optional first message with standing instructions/persona; not part of the back-and-forth, more like stage directions


**BUT WAIT there is more!!**

**OUTPUT FORMAT**
...T_T...


## **Extracting the output**

Different models have different ways to hand back their output, so we need to know the model type to know where to reach for the answer. T-T ...let's put
some more seasoning on all of this.

Remember model types?? yes me either...

This comes straight from the architecture,
i.e. **where the answer starts**:

* **Decoder only Models** (Llama, GPT, Mistral, TinyLlama):
   * One single stream token after tokenm the model just continues the text.
   * So the answer starts **at the end of the prompt**, glued on.
   * `output[0]` = the **entire prompt + the answer**.
      * So we must **slice off the input** first: `output[0][input_len:]`,
      * if no, we decode the whole essay back and the "pass"/"fail" check matches a word inside the essay, not the model's actual answer...yes easy peasy...

* **Encoder/decoder** Models (T5, BART, FLAN-T5):
   * Remember here, two components the encoder reads the input, a separate decoder writes.
   * So the answer starts fresh, the input never appears in the output.
   * `output[0]` = **only the newly generated text** decode it directly, no need for slicing.


**Why I care about this**
* Because it bit me T-T
* I was getting `-1`s in `y_pred` and a classification report that made no sense
   * That `-1` is my "unparseable" bucket fires when the decoded text contains neither "pass" nor "fail". And a pile of `-1`s is almost always the extraction going wrong for exactly the reason above: if I forget to slice off the prompt on a decoder-only model, I'm scanning the whole essay text instead of the shortanswer,
   * So the parse is unreliable and the metrics go haywire.
   * Problems with prompt echo, model just echoing back the prompt, or parroting the text rules
     * Instead the generated output extracting is checkking the first match, but the first  match is the model repeating the prompt not answering it

So we need to match the extraction to the architecture. `-1`s aren't "the model
is bad", it is can be that I'm reading the output from the wrong place.

(That is.... help! send a search party with a helicopter and half the state troopers; I am so lost, and we havent even start with quantization T-T)

#### **Testing llama function**

**Explain the code**

* **max_lenghth = 2048** or lower,  this is the ceiling
  * This is the model context windown, related to the max number of tokens the model can pay attention to in one sequence
     * TinyLlama was pretrained, it learned positional embeddings, the model needs to know where each token sits in the sequence (position 1, position 2, ... position 2048)
     * It has no learned representation for position 2048 and beyond
  * This include the total input+output= these cannot passed 2048 tokens
    * This is importnat this 2048 has to cover instructions+essay + everything. So need to estimate like how long are the essay, the instructions, and the expected llm output
      * It has to be within the model max_lenghth
  * This is also do get data truncated, increasing this number beyond the model max makes the model instable
  * check the output and maybe lets check how many essay exceed teh 2000 tokens

In [ ]:
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

X_test_prompts["prompt_tokens"] = X_test_prompts["text"].apply(count_tokens)

print(X_test_prompts["prompt_tokens"].describe())
print("\nlongest prompt :", X_test_prompts["prompt_tokens"].max(), "tokens")
print("over 2048      :", (X_test_prompts["prompt_tokens"] > 2048).sum())

Keep in mind that to this we need to add the model response
  * In our case we put nes tokens to 10 so we still are within the max_token

In [ ]:
# Length of just the essay
test_df["essay_tokens"] = test_df["full_text"].apply(count_tokens)
print(test_df["essay_tokens"].describe())

In [ ]:
#deciding where to slide the prompt and teh llm answer, easy...
print(repr(X_test_prompts.iloc[0]["text"]))

One essay at a time
* The loop processes one index per iteration
  *  for i in tqdm(range(len(test)))
  *  test.iloc[i], the [i] picks out exactly one row
  *  Padding False
  


In [ ]:
# Quick zero-shot check on 50 VAL essays (test stays sealed for final numbers)
val_sample   = X_val_prompts.iloc[:50].reset_index(drop=True)
y_val_sample = y_val.values[:50]

y_pred, y_generated = predict_decoder_only(val_sample,
                                           model,
                                           tokenizer, verbose=True)

In [ ]:
# ── Evaluate against VAL labels ──
assert len(y_pred) == len(y_val_sample)

# drop unparseable (-1) predictions
valid = [i for i, p in enumerate(y_pred) if p != -1]
yt = y_val_sample[valid]
yp = [y_pred[i] for i in valid]

print(f"Parsed: {len(valid)}/{len(y_pred)}")
print(f"Accuracy: {accuracy_score(yt, yp):.4f}")
print(classification_report(yt, yp, target_names=["Fail", "Pass"], zero_division=0))
print(confusion_matrix(yt, yp))
print(f"Predicted-Pass fraction: {(pd.Series(yp)==1).mean():.3f}  (true rate: {(pd.Series(yt)==1).mean():.3f})")

Results
* This model 1.1B model can't do this task....

In [ ]:
# predict on first 10 test essays only
test_sample = X_test_prompts.iloc[:50].reset_index(drop=True)
y_true_sample = y_true[:50]          #  this is what was missing

y_pred, y_generated = predict_decoder_only(test_sample, model, tokenizer)

In [ ]:
results_df = pd.DataFrame({
    "y_true"    : y_true_sample,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true_sample[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    labels = [0, 1]
    target_names = ["Fail", "Pass"]
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=labels, target_names=target_names, zero_division=0
    ))
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=labels)
    for label, row in zip(target_names, cm):
        print(f"True {label:<5}: {row}")
else:
    print("No valid predictions to evaluate.")

### Finetuning

In [ ]:
# TinyLlama loading A100 GPUedition
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token     # TinyLlama: no dedicated pad alias eos (correct here)
tokenizer.padding_side = "right"              # training mode (left for inference later)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,   # A100 GPU, fp32 was the CPU setting
    device_map="auto",
)
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False

In [ ]:
#!pip uninstall -y pyarrow datasets trl -q
#!pip install --no-cache-dir --force-reinstall pyarrow datasets trl transformers accelerate peft -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
from datasets import Dataset

In [ ]:
train_hf = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in train_balanced.iterrows()])
val_hf   = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in val_df.iterrows()])
print(repr(train_hf[0]["prompt"][-70:]))   # want TinyLlama/Zephyr style: ...</s>\n<|assistant|>\n

In [ ]:
#!pip install -U torchao

In [ ]:
#A100 GPU
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()          # trainable parameters

sft_config = SFTConfig(
    output_dir="./tinyllama-lora-balanced",
    num_train_epochs=1,
    per_device_train_batch_size=16,          # 1.1B on 80GB go big or go home
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,           # effective batch = 16
    warmup_steps=50,
    learning_rate=2e-4,                      # standard LoRA rate; small models tolerate it
    max_grad_norm=0.3,
    fp16=False, bf16=True,                   # A100: bf16, only bf16=True with a100 GPU!
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    report_to="none",
    max_length=2000,                         # TinyLlama's real ceiling is 2048, remember!
    completion_only_loss=True,
    optim="adamw_torch",                     # no memory pressure today we rich
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf.select(range(250)),
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
# Save locally
ADAPTER_PATH = "./tinyllama-lora-balanced-adapter"

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Saved locally: {ADAPTER_PATH}")
print(f"Files: {os.listdir(ADAPTER_PATH)}")

In [ ]:
# Backup to Drive dont forget
import shutil, os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

LOCAL_ADAPTER_PATH = "./tinyllama-lora-balanced-adapter"
DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/tinyllama-lora-balanced-adapter"

assert os.path.exists(LOCAL_ADAPTER_PATH), "No local adapter — did the save cell run?"

if os.path.exists(DRIVE_ADAPTER_PATH):
    shutil.rmtree(DRIVE_ADAPTER_PATH)
    print("Removed old version from Drive")

shutil.copytree(LOCAL_ADAPTER_PATH, DRIVE_ADAPTER_PATH)
print(f"   Backed up to Drive: {DRIVE_ADAPTER_PATH}")
print(f"   Files: {os.listdir(DRIVE_ADAPTER_PATH)}")

In [ ]:
tokenizer.padding_side = "left"          # flip back from training's "right"
model_ft = trainer.model                 # best checkpoint (load_best_model_at_end)
model_ft.eval()
model_ft.config.use_cache = False

smoke = X_val_prompts.iloc[:1].reset_index(drop=True)
_, g = predict_decoder_only(smoke, model_ft, tokenizer)
print(repr(g[0]))    # want a clean 'pass' or 'fail'

In [ ]:
y_pred, y_generated = predict_decoder_only(
    X_test_prompts.reset_index(drop=True), model_ft, tokenizer
)

# save straight to Drive before anything else
pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "generated": y_generated}) \
  .to_csv("/content/drive/MyDrive/tinyllama_finetuned_test_results.csv", index=False)

In [ ]:
valid = [i for i, p in enumerate(y_pred) if p != -1]
yt = y_true[valid]; yp = [y_pred[i] for i in valid]

print(f"Parsed: {len(valid)}/{len(y_pred)}")
print(f"Accuracy: {accuracy_score(yt, yp):.4f}")
print(classification_report(yt, yp, target_names=["Fail", "Pass"]))
print(confusion_matrix(yt, yp))
print("Predicted-Pass fraction:", (pd.Series(yp)==1).mean(), " (true rate 0.42)")

### Google model 250M

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [ ]:
#model loading
# no quantization
model_name = "google/flan-t5-base" #250m smaller

tokenizer  = T5Tokenizer.from_pretrained(model_name)
model      = T5ForConditionalGeneration.from_pretrained(
                 model_name,
                 #torch_dtype=torch.float16, #we could do 32 but lets do 16
                 torch_dtype=torch.float32,
                 device_map="auto"
             )

In [ ]:

#model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Vocab size     :", len(tokenizer))
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

# Sanity-check: confirm Pass/Fail tokens exist in vocabulary
pass_id = tokenizer.encode("Pass", add_special_tokens=False)
fail_id = tokenizer.encode("Fail", add_special_tokens=False)
print(f"\nToken check:")
print(f"  'Pass' → token id(s): {pass_id}")
print(f"  'Fail' → token id(s): {fail_id}")

In [ ]:
print(model)
# what element i would lookhere the attention layers
# the max features here is 2048, this is important to know see below


In [ ]:
print(model.config)
#here i will check dtype if quant worked it showed here
# check the model type   "is_decoder": false,
# "is_encoder_decoder": true,
#"model_type": "t5",
# "n_positions": 512, this is the max length for this model

In [ ]:
print(model.get_memory_footprint() / 1e9, "GB")
#model size increases from float16 to 32

Free genertion approach to get output

* This is no a good model to classify essay it is too small the max lengh is too narrow

In [ ]:
def predict_t5(test, model, tokenizer):
    y_pred      = []
    y_generated = []

    model.eval() # disable training only behavior

   #it is a loop that goes through
    for i in tqdm(range(len(test))): #this is for the process bar
        prompt = test.iloc[i]["text"]

        # truncate to 512 tokens  T5's hard limit

        #the prompt needs to be converted into tokensID (numeric)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512, #if essays are longer than 512 tokens but the rest bye bye, many will go bye bye
            padding=False #one essay at a time, no multiple essays so no ned to pad multiple different essays
        ).to(model.device) #find the model, device it is in the cpu or gpu
                           #we are moving the tokenized prompt to th model


        #no gradient tracking we are not training
        with torch.no_grad():
            # T5 uses .generate() directly, NOT pipeline("text-generation")
            # different models have different way to extract their generated output
            # this tells the model to generate the tokens
             #the models tht needs pipeline ("text-generate") is for decoder only models
             #such s GPT adn llama wher input output shar th same workflow
             #T5 is a encoder/decoder model it understand the input stage and the output stage diffrently
            outputs = model.generate(
                **inputs,
                max_new_tokens=5, #up to 5 tokens
            )

        #this generate the token id for the first batch only 0
        # tokenizer.decode() token id back to human readable text
        #skip_special get mor clean tokens
        # strip() remove leading whitspace and everything lower
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()


        #for the y generated if the generated token has the world pass predict 1
        # fail as 0,  otherwise mark it as -1
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {y_pred[-1]}")

    return y_pred, y_generated




In [ ]:
#  run on val set first
val_sample   = X_val_prompts.iloc[:50].reset_index(drop=True)
y_val_sample = y_val.values[:50]

y_pred, y_generated = predict_t5(val_sample, model, tokenizer)

In [ ]:
results_df = pd.DataFrame({
    "y_true"    : y_val_sample,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask     = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid   = y_val_sample[valid_mask]
y_pred_valid   = [y_pred[i] for i in valid_mask]

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0,1], target_names=["Fail","Pass"], zero_division=0
    ))

In [ ]:
# run on full test set
y_pred, y_generated = predict_t5(X_test_prompts, model, tokenizer)

#test
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

### Gemma 2 2B

* Differences between gated and not gated model sweet beans...
* I need to activate my HF key

## Gated vs. Non-Gated Models...

Not every model on huggingface is free to just download and go. Some are
**gated**, error of (a 401) and
lose 15 minutes wondering what is wrong. (I did. T-T)

### What's the difference?

* **Non-gated (open) models**: anyone can download them, no login, no token,
  no permission. Just call `from_pretrained(...)` and it works.
    - Examples we used: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, `google/flan-t5-large`
    - Also ungated: Qwen, SmolLM, most community models

* **Gated models**: the owner (usually a big company) requires you to
  **agree to a license first** before the model can be downloaded. The weights are still
  free, but access is controlled.
    - Examples: `google/gemma-2-2b-it` (Google), `meta-llama/*` (Meta)
    - Why gated? License terms, acceptable-use policies, "know who's using it"

### The gotcha: TWO things are required, not one
(dont be me)

1. **Accept the license in the browser (one-time).**
   Go to the model page and while logged in, and click to accept the terms. Access is usually instant.
     * This grants YOUR ACCOUNT access to this model.

2. **Authenticate the notebook with your unique HF token.**
   The token proves *who you are* and proves *you're allowed in*.
     * You need BOTH. Token without license accepted = still 401.
     * License accepted without token = notebook doesn't know who you are = 401.



In [ ]:
#interactive logging for the workshop
from huggingface_hub import login
login()   # paste your token

In [ ]:
# Now this is a gated Model, #no Quant but it is really slow using CPU, moving to to L4GPU to get thigs faster
model_name = "google/gemma-2-2b-it"

# Gemma is decoder only
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token   # Gemma, like Llama, has no dedicated PAD
tokenizer.padding_side = "left"             # left-pad for inference, but we not batching really we dong one essay at a time so padding is not needed either but lets keep it

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,   # float32 = CPU safe; switch to float16 if on a T4 GPU
    #HF_TOKEN=HF_TOKEN,
    device_map="auto",           # let HF place it on GPU if available
)

model.config.pad_token_id = tokenizer.eos_token_id # we can star using more fancy coding
# this model is not working

In [ ]:
#HF_TOKEN = "B-Llama-3.1-8B-Access" this way is not working

In [ ]:
print(model)
# what element i would lookhere the attention layers
# the max features here is 8192, this is important to know see below
# "max_position_embeddings": 8192

eos
* "eos_token_id": [1, 107]
   * token i the classic en of everything token
   * 107 end of turn, the end of turn token here in GEMM is registered as EOS

pad
  * "pad_token_id": 1. Gemma has a real, dedicated pad token, a single integer.
  * tokenizer.pad_token_id returns a clean 1, exactly the single value generate wants.
    * So for gemma since it has its PAD
      * ad_token_id=tokenizer.pad_token_id
         * hands to generate it 1: correct, clean.

We will not use here
* pad_token_id=tokenizer.eos_token_id
   * We are not combining pad with eos why? Gemma is fancy it has both pad token and eos tokens
     * risks handing generate the list [1, 107]

In [ ]:
print(model.config)
#here i will check dtype if quant worked it showed here
# some elements to look
# attention layers
#"dtype": "float32"
# PAD token as "pad_token_id": 1
# EOS token as
# "eos_token_id": [
#    1,
#    107
#  ]



In [ ]:
print(model.get_memory_footprint() / 1e9, "GB")
#model size increases from float16 to 32

In [ ]:
#model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Vocab size     :", len(tokenizer))
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

# Sanity-check: confirm Pass/Fail tokens exist in vocabulary
pass_id = tokenizer.encode("Pass", add_special_tokens=False)
fail_id = tokenizer.encode("Fail", add_special_tokens=False)
print(f"\nToken check:")
print(f"  'Pass' → token id(s): {pass_id}")
print(f"  'Fail' → token id(s): {fail_id}")

In [ ]:

#model.config.pad_token_id = tokenizer.eos_token_id



Gemma has 2 eos tokens
* "eos_token_id": [1, 107]
   * remember this from model info
So
 * tokenizer.eos_token_id may return a list, not a single integer
 * .generate wants a single int for pad_token_id, and handing it a list can throw an error
    * this part here pad_token_id=tokenizer.eos_token_id
    * How do i know this.....errors here, errors there, erros everywhere

    

In [ ]:
def predict_gemma(test, model, tokenizer):
    y_pred      = []
    y_generated = []

    model.eval()  # disable training only behavior (we not training here)

    # loop through every prompt in the test set
    for i in tqdm(range(len(test))):  # tqdm gives the progress bar, really helpful

        messages = [{"role": "user", "content": test.iloc[i]["text"]}]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        # convert the prompt string into token IDs (numeric)
        # we are giving all of this to our model in a language it can understand (tokenize)
        inputs = tokenizer(
            prompt,             #this is the prompt
            return_tensors="pt",  #tensor
            truncation=True,      # cut, not sure how this affect the evaluation
            max_length=8192,   # Llama handles longer context than T5's 512, anything more 2048 bye bye
            padding=False      # one essay at a time, nothing to pad against, less problems
        ).to(model.device)     # move the tokenized prompt to wherever the model lives (cpu/gpu)

        # remember how long the INPUT was, in tokens
        # decoder-only models return input + answer glued together,
        # so we need this to cut the input back off afterwards

        input_len = inputs["input_ids"].shape[1]

        # no gradient tracking, we are not training
        with torch.no_grad():
            # Llama GPT are decoder only no not forget this so --->>>input and output share one stream,
            # so .generate() APPENDS the answer onto the prompt tokens
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,         # only need a word or two ("Pass"/"Fail")
                do_sample=False,           # greedy = deterministic, reproducible in a workshop
                pad_token_id=tokenizer.pad_token_id,  # read above why
            )
        # Where the answerrrrr
        # The key step here: keep only the tokens AFTER the prompt.
        # outputs[0] = [ ...prompt tokens... , ...new answer tokens... ]
        # so lets slice from input_len onward to get ONLY what the model added.

        new_tokens = outputs[0][input_len:]

        # turn those new token IDs back into text
        # skip_special_tokens drops <|assistant|>, </s>, etc.
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

        y_generated.append(generated)

        # map the text answer to a label, So
        # contains "pass" -> 1, contains "fail" -> 0, neither -> -1 (unparseable problems)
        if "pass" in generated: y_pred.append(1)
        elif "fail" in generated: y_pred.append(0)
        else: y_pred.append(-1)

        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  → → {y_pred[-1]}")

    return y_pred, y_generated

In [ ]:
i = 0
messages = [{"role": "user", "content": val_sample.iloc[i]["text"]}]
p = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#we can see the model maker, i mean if you want to see it
print(repr(p[-120:]))   # tail should show ...<start_of_turn>model\n
ins = tokenizer(p, return_tensors="pt").to(model.device)
out = model.generate(**ins, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.pad_token_id)
print("SLICED:", repr(tokenizer.decode(out[0][ins['input_ids'].shape[1]:], skip_special_tokens=True)))

In [ ]:
def predict_t5(test, model, tokenizer):
    y_pred      = []
    y_generated = []

    model.eval()
    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]

        # truncate to 512 tokens — T5's hard limit
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=False
        ).to(model.device)

        with torch.no_grad():
            # T5 uses .generate() directly, NOT pipeline("text-generation")
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
            )

        generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {y_pred[-1]}")

    return y_pred, y_generated

In [ ]:
## Val first

In [ ]:
# remember this model is at high capacity of float32
# 30m to run 100 using CPU
# 3m to run 100 using L4GPU
# run on val set first
val_sample   = X_val_prompts.iloc[:100].reset_index(drop=True)
y_val_sample = y_val.values[:100]

y_pred, y_generated = predict_gemma(val_sample, model, tokenizer)

results_df = pd.DataFrame({
    "y_true"    : y_val_sample,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask     = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid   = y_val_sample[valid_mask]
y_pred_valid   = [y_pred[i] for i in valid_mask]

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0,1], target_names=["Fail","Pass"], zero_division=0
    ))

In [ ]:
#testing

In [ ]:
# run on full test set
# will take too long
y_pred, y_generated = predict_gemma(X_test_prompts, model, tokenizer)

#test
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

#QUANTIZATION

* What is quant?? well it is a technique that compress the model, it reduces llm by converting/changing their weights activations. Quantified Weight activation from maybe high precision formats like FP32 or FP16 to lower representation like INT8 (8bit quant) or INT4 (4bit quant)

* Models are usually represented in bfloat 16bits at least in huggingface (bfloat 16 is huge is one of the latest, so a100 GPU and up)

The precision of model from Hugging Face may vary depending on how the model was stored (e.g., FP32 weights or bP16 weights), but i do think most models are bf16
  * To run BF16 A100 GPU are required, remember bf16 is one of the latest precision
    * https://huggingface.co/docs/optimum/en/concept_guides/quantization
    *  https://huggingface.co/docs/hub/en/gguf#quantization-types

### Mistral 7B **4bit**

Why quant Mistral?
* Mistral-7B in float16 is ~14 GB
   * too big for a T4's 16 GB alongside activations. 4-bit quantization (load_in_4bit) compresses the weights to 4 GB, so it fits. This is the only way to run a 7B on a T4 we cannot run this model using cpu
   * nf4 + double quant + fp16 compute are the standard QLoRA-style settings
      * nf4 is the 4-bit number format optimized for neural-net weights
      * double-quant squeezes a bit more,
      * compute power stays in fp16 so the math is still reasonably precise

In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers

In [ ]:
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)

In [ ]:
# 8m GPU 4T
# 1m GPU L4
# Load Mistral 7B with 4-bit quantization (required for T4 16GB)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # see we are float16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    #HF_TOKEN= HF_TOKEN,
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

In [ ]:
print(model)
# what element i would lookhere the attention layers
# the max features here is 8192, this is important to know see below
# "max_position_embeddings": 8192

eos
* "eos_token_id": 2

pad
  * "pad_token_id": 2
For Mistral the eos and pad point to the same token. So here
 * tokenizer.pad_token = tokenizer.eos_token
 * pad_token_id=tokenizer.pad_token_id both resolve to the same value and work without conflict.

We dont have to worry about max token here
*  "max_position_embeddings": 32768,

In [ ]:
print(model.config)
#here i will check dtype if quant worked it showed here
# some elements to look
# attention layers
#"dtype":
# PAD token as "pad_token_id":
# EOS token as "eos_token_id"



In [ ]:
print(model.get_memory_footprint() / 1e9, "GB")
#model size increases from float16 to 32

In [ ]:
#model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Vocab size     :", len(tokenizer))
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

# Sanity-check: confirm Pass/Fail tokens exist in vocabulary
pass_id = tokenizer.encode("Pass", add_special_tokens=False)
fail_id = tokenizer.encode("Fail", add_special_tokens=False)
print(f"\nToken check:")
print(f"  'Pass' : token id(s): {pass_id}")
print(f"  'Fail' : token id(s): {fail_id}")

In [ ]:
def predict_mistral(test, model, tokenizer):
    y_pred      = []
    y_generated = []

    model.eval()  # disable training only behavior (we not training here)

    # loop through every prompt in the test set
    for i in tqdm(range(len(test))):  # tqdm gives the progress bar, really helpful

        messages = [{"role": "user", "content": test.iloc[i]["text"]}]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        # convert the prompt string into token IDs (numeric)
        # we are giving all of this to our model in a language it can understand (tokenize)
        inputs = tokenizer(
            prompt,             #this is the prompt
            return_tensors="pt",  #tensor
            truncation=True,      # cut, not sure how this affect the evaluation
            max_length=10000,   # Llama handles longer context than T5's 512, anything more 2048 bye bye
            padding=False      # one essay at a time, nothing to pad against, less problems
        ).to(model.device)     # move the tokenized prompt to wherever the model lives (cpu/gpu)

        # remember how long the INPUT was, in tokens
        # decoder-only models return input + answer glued together,
        # so we need this to cut the input back off afterwards

        input_len = inputs["input_ids"].shape[1]

        # no gradient tracking, we are not training
        with torch.no_grad():
            # Llama GPT are decoder only no not forget this so --->>>input and output share one stream,
            # so .generate() APPENDS the answer onto the prompt tokens
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,         # only need a word or two ("Pass"/"Fail")
                do_sample=False,           # greedy = deterministic, reproducible in a workshop
                pad_token_id=tokenizer.eos_token_id,
            )
        # Where the answerrrrr
        # The key step here: keep only the tokens AFTER the prompt.
        # outputs[0] = [ ...prompt tokens... , ...new answer tokens... ]
        # so lets slice from input_len onward to get ONLY what the model added.

        new_tokens = outputs[0][input_len:]

        # turn those new token IDs back into text
        # skip_special_tokens drops <|assistant|>, </s>, etc.
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

        y_generated.append(generated)

        # map the text answer to a label, So
        # contains "pass" -> 1, contains "fail" -> 0, neither -> -1 (unparseable problems)
        if "pass" in generated: y_pred.append(1)
        elif "fail" in generated: y_pred.append(0)
        else: y_pred.append(-1)

        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  → → {y_pred[-1]}")

    return y_pred, y_generated

In [ ]:
#sample of 100
val_sample   = X_val_prompts.iloc[:100].reset_index(drop=True)
y_val_sample = y_val.values[:100]

y_pred, y_generated = predict_mistral(val_sample, model, tokenizer)

results_df = pd.DataFrame({
    "y_true"    : y_val_sample,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask     = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid   = y_val_sample[valid_mask]
y_pred_valid   = [y_pred[i] for i in valid_mask]

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0,1], target_names=["Fail","Pass"], zero_division=0
    ))

In [ ]:
# The whole essays test
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

#### **LoRA Finetuning**

* Yes, we've arrived at the eye of the storm, welcome to fine-tuning!  Now the fun begins. Let's add more spice to the mix!

In [ ]:
!pip install -U peft trl bitsandbytes accelerate transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

Model
* HF model card here

In [ ]:
model_name = "microsoft/phi-2"

# Load in 4bit, cuz we poor and dont have A100 GPU today
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # right padding for training (left for inference, dont forget!!)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Base model loaded on:", next(model.parameters()).device)
print("Memory (MB)         :", round(model.get_memory_footprint() / 1e6, 1))

In [ ]:
print(model)
# what element i would lookhere the attention layers
# the max features here is 8192, this is important to know see below
# "max_position_embeddings": 8192

eos
*   eos_token_id": 50256

pad
*   pad_token_id": 50256

In [ ]:
print(model.config)
#here i will check dtype if quant worked it showed here
# some elements to look
# attention layers
#"dtype": "float32"
# PAD token as "pad_token_id": 1
# EOS token as
# "eos_token_id": [
#    1,
#    107
#  ]



In [ ]:
print(model.get_memory_footprint() / 1e9, "GB")
#model size increases from float16 to 32

In [ ]:
#model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Vocab size     :", len(tokenizer))
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

# Sanity-check: confirm Pass/Fail tokens exist in vocabulary
pass_id = tokenizer.encode("Pass", add_special_tokens=False)
fail_id = tokenizer.encode("Fail", add_special_tokens=False)
print(f"\nToken check:")
print(f"  'Pass' → token id(s): {pass_id}")
print(f"  'Fail' → token id(s): {fail_id}")

##Microsoft Phi-3

Phi 2 ~4B
* It is a base model not a instruct model, so what we have used previously (on our previous episodes) of tokenizer.apply_chat_template()
  * remember this to get us  the chat template that belongs to a model that is for instruct model...Phi is a base model
  * Phi does not have a chat template
  * IM not sure if Phi is useful for our tasks
So,
maybe lets not do
 * model_name = "microsoft/phi-2"
 * we can use phi  instruct version

In [ ]:
!pip install -U transformers accelerate

Please note that we are not quantizing the model.

Model info
*

In [ ]:
#model_name = "microsoft/phi-2"

model_name = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    #trust_remote_code=True,
)



eos
*   "eos_token_id": 32000

pad
*   pad_token_id": 32000

Max token 131072
* wow that's like 130K context. Our ~700-token prompts are microscopic against it, we dont need to worry about ever truncates

In [ ]:
print(model)

In [ ]:
print(model.config)
#here i will check dtype if quant worked it showed here
# some elements to look
# attention layers
#"dtype": "float32"
# PAD token as "pad_token_id":
# EOS token as "eos_token_id":


In [ ]:
print(model.get_memory_footprint() / 1e9, "GB")
#model size increases from float16 to 32

In [ ]:
#model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Vocab size     :", len(tokenizer))
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

# Sanity-check: confirm Pass/Fail tokens exist in vocabulary
pass_id = tokenizer.encode("Pass", add_special_tokens=False)
fail_id = tokenizer.encode("Fail", add_special_tokens=False)
print(f"\nToken check:")
print(f"  'Pass' → token id(s): {pass_id}")
print(f"  'Fail' → token id(s): {fail_id}")

In [ ]:
def predict_phi3(test, model, tokenizer):
    y_pred      = []
    y_generated = []

    model.eval()  # disable training only behavior (we not training here)

    # loop through every prompt in the test set
    for i in tqdm(range(len(test))):  # tqdm gives the progress bar, really helpful

        messages = [{"role": "user", "content": test.iloc[i]["text"]}]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        # convert the prompt string into token IDs (numeric)
        # we are giving all of this to our model in a language it can understand (tokenize)
        inputs = tokenizer(
            prompt,             #this is the prompt
            return_tensors="pt",  #tensor
            truncation=True,      # cut, not sure how this affect the evaluation
            max_length=10000,   # Llama handles longer context than T5's 512,
            padding=False      # one essay at a time, nothing to pad against, less problems
        ).to(model.device)     # move the tokenized prompt to wherever the model lives (cpu/gpu)

        # remember how long the INPUT was, in tokens
        # decoder-only models return input + answer glued together,
        # so we need this to cut the input back off afterwards

        input_len = inputs["input_ids"].shape[1]

        # no gradient tracking, we are not training
        with torch.no_grad():
            # Llama GPT are decoder only no not forget this so --->>>input and output share one stream,
            # so .generate() APPENDS the answer onto the prompt tokens
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,         # only need a word or two ("Pass"/"Fail")
                do_sample=False,           # greedy = deterministic, reproducible in a workshop
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False,
            )
        # Where the answerrrrr
        # The key step here: keep only the tokens AFTER the prompt.
        # outputs[0] = [ ...prompt tokens... , ...new answer tokens... ]
        # so lets slice from input_len onward to get ONLY what the model added.

        new_tokens = outputs[0][input_len:]

        # turn those new token IDs back into text
        # skip_special_tokens drops <|assistant|>, </s>, etc.
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

        y_generated.append(generated)

        # map the text answer to a label, So
        # contains "pass" -> 1, contains "fail" -> 0, neither -> -1 (unparseable problems)
        if "pass" in generated: y_pred.append(1)
        elif "fail" in generated: y_pred.append(0)
        else: y_pred.append(-1)

        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  → → {y_pred[-1]}")

    return y_pred, y_generated

In [ ]:
# remember this model is at high capacity of float32
# 30m to run 100 using CPU
# 3m to run 100 using L4GPU
# run on val set first
val_sample   = X_val_prompts.iloc[:100].reset_index(drop=True)
y_val_sample = y_val.values[:100]

y_pred, y_generated = predict_phi3(val_sample, model, tokenizer)

results_df = pd.DataFrame({
    "y_true"    : y_val_sample,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask     = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid   = y_val_sample[valid_mask]
y_pred_valid   = [y_pred[i] for i in valid_mask]

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0,1], target_names=["Fail","Pass"], zero_division=0
    ))

In [ ]:
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

#### LoRA finetuning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U peft trl bitsandbytes accelerate transformers
!pip install -U transformers accelerate

Careful with the target modules
* target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"] these are the one for phi 2
* different models will have different targets

In [ ]:
#find the model target
import torch.nn as nn
linear_names = {name.split(".")[-1] for name, module in model.named_modules()
                if isinstance(module, nn.Linear)}
print(linear_names)

LoRA parameters
* r=   common values are 8, 16, 32,64
  * maybe 8
  * LoRa freezes a small matric and learns a small correct to it, writing as the product of two small matrices

* lora_alpha = 32
  * this is a scaling factor the correction gets multiplided by alpha/r before it is addded to the frozen weights
  * in our example 32/16 = 2
  * this is about how strong the contribution is relateive to the base model

* lora_dropout= 0.05
  * this is a regularization. During training, 5% of the adapters get shut down, this is to minimize overfitting and memorizing the data

* bias ="none"
  * dont train on bias terms. Linear weight sometime has biases

* task_type=TaskType.CAUSAL_LM
  * this is about telling PEFT what model it is so the adapter knows if it is a decoder only model, tells the adapter the type of model

* target_modules= [...]
  * where the adapters go, needs to checks this by model
    * check the attention layers
      * models decides what to pay attention too this is related to
        like q k v project, o_project
        * gate_up_proj adn down_proj
        * these are many times architecture specific

(There are other parameters see: [link text](https://4d9506f4.isolation.zscaler.com/profile/99dc41b0-1dad-4477-abaf-98852b346d52/zia-session/?controls_id=5ce955a4-1151-4a37-87ed-461c122b389e&region=was&tenant=9b5cea868a14&user=f5f5d245a2f0743f905306cd843b572624a923ce78e9fcb89f3e8f33ee0116f0&original_url=https%3A%2F%2Fhuggingface.co%2Fdocs%2Fpeft%2Fen%2Fpackage_reference%2Flora&key=sh-1&hmac=fc1517c6fb75016d771473ef1189709d2ca3b547e7e1ab4e403ebbca03f98aa2)

In [ ]:
# Part 1 model and QLoRA parameters
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch
import gc

MODEL_NAME   = "microsoft/Phi-3.5-mini-instruct"
ADAPTER_PATH = "./phi3-lora-balanced-adapter"

# 4-bit quantization: float16 required for T4 (no bfloat16 support)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,                       # higher more correction, more trainable mor memory and more overfitting risk
                                # lower more constrained
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
)

In [ ]:
X = sample.drop(columns=["holistic_essay_score", "binary_score"])
y = sample["binary_score"].copy()          # binary 0/1 instead of 1 to 6, th 1 to 6 was not working at all

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train):,}  ({len(X_train)/len(sample)*100:.1f}%)")
print(f"Val   : {len(X_val):,}   ({len(X_val)/len(sample)*100:.1f}%)")
print(f"Test  : {len(X_test):,}   ({len(X_test)/len(sample)*100:.1f}%)")
print("\nClass distribution (stratification check):")
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = y_split.value_counts().rename({0: "Fail", 1: "Pass"})
    print(f"  {name}: {counts.to_dict()}")

In [ ]:
#Part 2
#better version to keep things consistent
def generate_test_prompt(row):
    return (
        f"You are an expert essay grader. Read the essay and respond with exactly one word.\n"
        f"Your response must be either the word Fail or Pass. No other words.\n\n"
        f"Rules:\n"
        f"- Fail: the essay is weak, underdeveloped, or below standard\n"
        f"- Pass: the essay is proficient, strong, or excellent\n\n"
        f"Prompt: {row['prompt_name']}\n"
        f"Task: {row['task']}\n"
        f"Essay: {row['full_text']}\n\n"
        f"Respond with one word only (Fail or Pass): "
    )


def make_prompt_completion(row, tokenizer):
    user_content = generate_test_prompt(row)   # same wording
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False, add_generation_prompt=True,
    )
    completion = "Pass" if row["binary_score"] == 1 else "Fail"
    return {"prompt": prompt, "completion": completion}

#Train
train_df = X_train.copy()
train_df["binary_score"] = y_train.values

# eval prompt
val_df = X_val.copy()
val_df["binary_score"] = y_val.values
X_val_prompts = pd.DataFrame(val_df.apply(generate_test_prompt, axis=1), columns=["text"])

test_df = X_test.copy()
test_df["binary_score"] = y_test.values
y_true = test_df["binary_score"].values
X_test_prompts = pd.DataFrame(test_df.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
#print(train_df["text"].iloc[0])

In [ ]:
# Part 3
fail_df = train_df[train_df["binary_score"] == 0]
pass_df = train_df[train_df["binary_score"] == 1]
min_class = len(pass_df)

train_balanced = pd.concat([
    fail_df.sample(n=min_class, random_state=42),
    pass_df,
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced distribution: {train_balanced['binary_score'].value_counts().to_dict()}")

In [ ]:
# Part 4: Build HuggingFace datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"    # right for training (flip back to "left" for inference)

train_hf = Dataset.from_list(
    [make_prompt_completion(r, tokenizer) for _, r in train_balanced.iterrows()]
)
val_hf = Dataset.from_list(
    [make_prompt_completion(r, tokenizer) for _, r in val_df.iterrows()]
)

print(f"Train : {len(train_hf)}")
print(f"Val   : {len(val_hf)}")
print("Columns:", train_hf.column_names)                      # should be ['prompt', 'completion']
print("Sample prompt ending:", repr(train_hf[0]["prompt"][-70:]))   # should end at <|assistant|>\n
print("Sample completion  :", repr(train_hf[0]["completion"]))      # should be 'Pass' or 'Fail'

In [ ]:
train_hf = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in train_balanced.iterrows()])
val_hf   = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in val_df.iterrows()])
print(train_hf[0])   # {'prompt': '<|user|>...<|assistant|>\n', 'completion': 'Pass'}

Huge Error
* fp16=True
* bf16=False
  * LoRA adapters loaded in bfloat16 (lora_A.default.weight such as torch.bfloat16, and "any bf16 trainable: True"), but fp16=True uses an fp16 gradient scaler that can't handle bf16 gradients. On a T4 (no bf16 support)
  * Cast the adapters to fp32

In [ ]:
# Part 5
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False                      #  checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)

#casting
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()   # so bf16 adapters to fp32 (standard QLoRA

# verify
print("any bf16 trainable:", any(p.dtype == torch.bfloat16 for p in model.parameters() if p.requires_grad))
model.print_trainable_parameters()                  # show small nonzero trainable

In [ ]:
#lens = [len(tokenizer.encode(t)) for t in train_hf["text"]]
#print("max:", max(lens), " | over 1024:", sum(l > 1024 for l in lens))
lens = [len(tokenizer.encode(ex["prompt"] + ex["completion"])) for ex in train_hf]
print("max:", max(lens), " | over 1024:", sum(l > 1024 for l in lens))

In [ ]:
!pip install trl

In [ ]:
for n, p in model.named_parameters():
    if p.requires_grad:   # the trainable LoRA/adapter params
        print(n, p.dtype)
        break
print("any bf16 trainable:", any(p.dtype==torch.bfloat16 for p in model.parameters() if p.requires_grad))

In [ ]:
import torch

In [ ]:
# Part 6 SFT config and  train

sft_config = SFTConfig(
    output_dir="./phi3-lora-balanced",
    num_train_epochs=1,                  # 1 epoch enough for binary labels, keeps runtime sane w/o fp16
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch = 8
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=False,                          # Ooff no GradScaler = no bf16 unscale crash error fix
    bf16=False,                          # T4 has no bf16 anyway
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,                      # eval passes are slow in full precision; less often
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    max_length=1024,                     # prompts ~700 tokens + label; nothing truncates
    completion_only_loss=True,           # loss only on the Pass/Fail completion
    optim="paged_adamw_8bit",
)

# cast LoRA adapters to fp32 right before training, stops the error
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf.select(range(150)),   # small eval slice: loss signal only, keeps eval passes quick
    processing_class=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()

In [ ]:
from transformers import GenerationConfig

In [ ]:
# Part 7: I think i lost count

# full test set with the finetuned model

# inference setup (the three things we always set)
tokenizer.padding_side = "left"          # was "right" for training
model_ft = trainer.model                 # best checkpoint (load_best_model_at_end)
model_ft.config.use_cache = False        # Phi cache bug

model_ft.generation_config = GenerationConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
# run YOUR predict function on the FULL test prompts
y_pred, y_generated = predict_phi3(X_test_prompts.reset_index(drop=True), model_ft, tokenizer)

In [ ]:
#Part 7 Evaluate the finetuned model (run AFTER training completes)
# this is val

#flip padding back for inference (was "right" for training)
tokenizer.padding_side = "left"

# ft for finetuned
model_ft = trainer.model
model_ft.eval()
model_ft.config.use_cache = False   # Phi remote-code cache bug: keep cache off for generate

# Same predict function, same eval prompts
y_pred_ft, y_gen_ft = predict_phi3(X_val_prompts.iloc[:100].reset_index(drop=True), model_ft, tokenizer)

y_val_sample = y_val.values[:100]
valid = [i for i, p in enumerate(y_pred_ft) if p != -1]
yt = y_val_sample[valid]; yp = [y_pred_ft[i] for i in valid]

print(f"Parsed: {len(valid)}/100")
print(f"Accuracy : {accuracy_score(yt, yp):.4f}   (baseline was 0.5800)")
print(classification_report(yt, yp, labels=[0,1], target_names=["Fail","Pass"], zero_division=0))
print("Predicted-Pass fraction:", (pd.Series(yp)==1).mean(), " (baseline was ~0.68, true rate 0.42)")

In [ ]:
# Final Evaluation

In [ ]:
# Part 7: val-100 eval of the finetuned model
tokenizer.padding_side = "left"           # flip back from training's "right"
model_ft = trainer.model
model_ft.eval()
model_ft.config.use_cache = False         # Phi cache bug

y_pred_ft, y_gen_ft = predict_phi3(
    X_val_prompts.iloc[:100].reset_index(drop=True), model_ft, tokenizer
)

y_val_sample = y_val.values[:100]
valid = [i for i, p in enumerate(y_pred_ft) if p != -1]
yt = y_val_sample[valid]; yp = [y_pred_ft[i] for i in valid]

print(f"Parsed: {len(valid)}/100")
print(f"Accuracy : {accuracy_score(yt, yp):.4f}   (zero-shot baseline: 0.5800)")
print(classification_report(yt, yp, labels=[0,1], target_names=["Fail","Pass"], zero_division=0))
print("Predicted-Pass fraction:", (pd.Series(yp)==1).mean(), " (baseline ~0.68, true rate 0.42)")

In [ ]:
# Save locally
ADAPTER_PATH = "./phi3-lora-balanced-adapter"


trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"   Saved locally: {ADAPTER_PATH}")
print(f"   Files: {os.listdir(ADAPTER_PATH)}")

In [ ]:
import shutil, os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

LOCAL_ADAPTER_PATH = "./phi3-lora-balanced-adapter"
DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/phi3-lora-balanced-adapter"


assert os.path.exists(LOCAL_ADAPTER_PATH), "No local adapter found did the save cell run?"

# replace any old version on Drive
if os.path.exists(DRIVE_ADAPTER_PATH):
    shutil.rmtree(DRIVE_ADAPTER_PATH)
    print("Removed old version from Drive")

shutil.copytree(LOCAL_ADAPTER_PATH, DRIVE_ADAPTER_PATH)
print(f"Backed up to Drive: {DRIVE_ADAPTER_PATH}")
print(f"   Files: {os.listdir(DRIVE_ADAPTER_PATH)}")

In [ ]:
# Cell 8 Load fine-tuned model for inference
gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "left"     # switch to left padding for inference

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()

X_test_prompts_phi2 = pd.DataFrame(
    test_df.apply(generate_phi2_test_prompt, axis=1), columns=["text"]
)
print(f"Test prompts : {len(X_test_prompts_phi2)}")
print("Last 50 chars:", repr(X_test_prompts_phi2["text"].iloc[0][-50:]))
# should end with '\nOutput:'

In [ ]:
# Step 1 imports
# from  new session
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, shutil, os
from google.colab import drive

# Step 2 copy from Drive to local disk
drive.mount("/content/drive")

DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/phi2-lora-balanced-adapter"  # exact folder name from your Drive
LOCAL_ADAPTER_PATH = "./phi2-lora-balanced-adapter"

if not os.path.exists(LOCAL_ADAPTER_PATH):
    shutil.copytree(DRIVE_ADAPTER_PATH, LOCAL_ADAPTER_PATH)
    print(f"Copied from Drive to local")
else:
    print(f" Already exists locally")

print(f"Files: {os.listdir(LOCAL_ADAPTER_PATH)}")


In [ ]:
# Step 3 load base model + adapter
MODEL_NAME = "microsoft/phi-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"     # left padding for inference

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
ft_model = PeftModel.from_pretrained(base_model, LOCAL_ADAPTER_PATH)
ft_model.eval()

print("✅ Model loaded")
print("   Device :", next(ft_model.parameters()).device)
print("   Memory :", round(ft_model.get_memory_footprint() / 1e6, 1), "MB")

In [ ]:
# Cell 9: Predict
def predict_phi2(test, model, tokenizer, max_input_tokens=512):
    y_pred, y_generated = [], []
    model.eval()

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=max_input_tokens, padding=False,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        generated  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        result = {1: "Pass", 0: "Fail", -1: "???"}[y_pred[-1]]
        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {result}")

    return y_pred, y_generated

y_pred, y_generated = predict_phi2(X_test_prompts_phi2, ft_model, tokenizer)

In [ ]:
# Cell 10: Evaluate
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

### Not usre

In [ ]:
# clean reload: native transformers Phi3 (has generate), attach saved adapter
import gc
del model_ft
try: del model, trainer
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    # NO trust_remote_code → library's built-in Phi3ForCausalLM, which HAS generate()
)

from peft import PeftModel
model_ft = PeftModel.from_pretrained(base, "./phi3-lora-balanced-adapter")
model_ft.eval()
model_ft.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained("./phi3-lora-balanced-adapter")
tokenizer.padding_side = "left"

# smoke test before any big run:
print(hasattr(model_ft, "generate"), hasattr(base, "generate"))

In [ ]:
data1 = X_val_prompts.iloc[:1].reset_index(drop=True) #one essay to check
y_predict, y_generated = predict_phi3(data1, model_ft, tokenizer)
print("raw output:", repr(y_generated[0]), "→ parsed:", y_predict[0])

In [ ]:
# ── FINAL: full test set (765 essays), finetuned model ──

y_pred, y_generated = predict_phi3(
    X_test_prompts.reset_index(drop=True), model_ft, tokenizer
)

In [ ]:
if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

In [ ]:
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.head(20).to_string())                                # sample, not all 765

# save the full table — do this NOW so the run is never lost
results_df.to_csv("phi3_finetuned_test_results.csv", index=False)

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}   (zero-shot baseline was 0.5800)")
    print("\nClassification Report:")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")
    print(f"\nPredicted-Pass fraction: {(pd.Series(y_pred_valid)==1).mean():.3f}   (true rate: {(pd.Series(y_true_valid)==1).mean():.3f})")
else:
    print("No valid predictions to evaluate.")

##### Retesting

In [ ]:
# Step 1: reload full 25k dataset
import kagglehub
import pandas as pd
import os

path   = kagglehub.dataset_download("nbroad/persaude-corpus-2")
scores = pd.read_csv(os.path.join(path, "persuade_2.0_human_scores_demo_id_github.csv"))

PASS_THRESHOLD       = 4
scores["binary_score"] = (scores["holistic_essay_score"] >= PASS_THRESHOLD).astype(int)

print(f"Full dataset     : {len(scores):,}")
print(f"Class distribution:\n{scores['binary_score'].value_counts()}")

In [ ]:
scores

In [ ]:
# Step 2: exclude essays already used in original 5k sample
used_ids = set(sample["essay_id_comp"])   # sample = your original 5k pull

fresh = scores[~scores["essay_id_comp"].isin(used_ids)].reset_index(drop=True)

print(f"Already used     : {len(used_ids):,}")
print(f"Fresh essays left: {len(fresh):,}")
print(f"Class distribution:\n{fresh['binary_score'].value_counts()}")

In [ ]:
#  Step 3: sample 500 fresh essays stratified by binary_score
from sklearn.model_selection import train_test_split

fresh_test, _ = train_test_split(
    fresh,
    train_size=500,
    random_state=99,               # different seed from original splits
    stratify=fresh["binary_score"]
)
fresh_test = fresh_test.reset_index(drop=True)

print(f"Fresh test size  : {len(fresh_test)}")
print(f"Class distribution:\n{fresh_test['binary_score'].value_counts()}")

In [ ]:
# Step 4: build prompts
X_fresh_prompts = pd.DataFrame(
    fresh_test.apply(generate_phi2_test_prompt, axis=1), columns=["text"]
)
y_fresh_true = fresh_test["binary_score"].values

print(f"Prompts ready    : {len(X_fresh_prompts)}")
print(f"Last 50 chars    : {repr(X_fresh_prompts['text'].iloc[0][-50:])}")
# should end with '\nOutput:'

In [ ]:
# ── Step 5: predict ───────────────────────────────────────────────────────────
y_fresh_pred, y_fresh_generated = predict_phi2(
    X_fresh_prompts, ft_model, tokenizer
)

In [ ]:
# Step 6: evaluate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

results_fresh = pd.DataFrame({
    "y_true"    : y_fresh_true,
    "y_pred"    : y_fresh_pred,
    "generated" : y_fresh_generated,
})
results_fresh["y_true_label"] = results_fresh["y_true"].map({1: "Pass", 0: "Fail"})
results_fresh["y_pred_label"] = results_fresh["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_fresh.to_string())

valid_mask   = [i for i, p in enumerate(y_fresh_pred) if p != -1]
y_true_valid = y_fresh_true[valid_mask]
y_pred_valid = [y_fresh_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_fresh_pred)}")
print(f"Unparseable : {len(y_fresh_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

#### Getting the Finetuned-Adapters

In [ ]:
# Step 1: load fresh test data
import kagglehub
import pandas as pd
import os

path   = kagglehub.dataset_download("nbroad/persaude-corpus-2")
scores = pd.read_csv(os.path.join(path, "persuade_2.0_human_scores_demo_id_github.csv"))

PASS_THRESHOLD         = 4
scores["binary_score"] = (scores["holistic_essay_score"] >= PASS_THRESHOLD).astype(int)

print(f"Full dataset: {len(scores):,}")

In [ ]:
# Step 2: sample 500 fresh essays
from sklearn.model_selection import train_test_split

fresh_test, _ = train_test_split(
    scores,
    train_size=1000,
    random_state=99,
    stratify=scores["binary_score"]
)
fresh_test = fresh_test.reset_index(drop=True)

print(f"Fresh test size: {len(fresh_test)}")
print(fresh_test["binary_score"].value_counts())

In [ ]:
#  Step 3: build prompts
X_fresh_prompts = pd.DataFrame(
    fresh_test.apply(generate_phi2_test_prompt, axis=1), columns=["text"]
)
y_true = fresh_test["binary_score"].values   #  this creates y_true

print(f"Prompts ready: {len(X_fresh_prompts)}")
print("Last 50 chars:", repr(X_fresh_prompts["text"].iloc[0][-50:]))
# should end with '\nOutput:'

In [ ]:
# ── Step 4: predict — this creates y_pred and y_generated ────────────────────
y_pred, y_generated = predict_phi2(X_fresh_prompts, ft_model, tokenizer)

In [ ]:
# Step 5: evaluate now y_true, y_pred, y_generated all exist
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

In [ ]:
# Cell 1: installs
!pip install -q transformers peft trl bitsandbytes accelerate datasets kagglehub

In [ ]:
# Cell 2: imports
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, shutil, os, pandas as pd
from google.colab import drive
from tqdm import tqdm

In [ ]:
# Cell 3: copy adapter from Drive to local
drive.mount("/content/drive")

# get exact folder name
print("phi2 folders in Drive:")
for f in os.listdir("/content/drive/MyDrive"):
    if "phi2" in f.lower() or "lora" in f.lower():
        print(f"  → '{f}'")

In [ ]:
# Cell 4: copy to local disk
DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/phi2-lora-balanced-adapter"  # confirm name from Cell 3
LOCAL_ADAPTER_PATH = "./phi2-lora-balanced-adapter"

if not os.path.exists(LOCAL_ADAPTER_PATH):
    shutil.copytree(DRIVE_ADAPTER_PATH, LOCAL_ADAPTER_PATH)
    print(f"Copied from Drive to local")
else:
    print(f"Already exists locally")

print(f"Files: {os.listdir(LOCAL_ADAPTER_PATH)}")
# should show: adapter_model.safetensors, adapter_config.json, tokenizer files

In [ ]:
# Cell 5: load base model + adapter
MODEL_NAME = "microsoft/phi-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"     # left padding for inference

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
ft_model = PeftModel.from_pretrained(base_model, LOCAL_ADAPTER_PATH)
ft_model.eval()

print(" Model ready")
print("   Device :", next(ft_model.parameters()).device)
print("   Memory :", round(ft_model.get_memory_footprint() / 1e6, 1), "MB")

In [ ]:
# Cell 6: prompt function
def generate_phi2_test_prompt(row):
    instruction = (
        f"Instruct: You are a reasonable essay grader.\n"
        f"Prompt name: {row['prompt_name']}\n"
        f"Task type: {row['task']}\n\n"
        f"Classify the essay as Pass or Fail.\n"
        f"Pass = proficient or above. Fail = developing or below.\n"
        f"Respond with one word only: Pass or Fail.\n\n"
        f"Essay: "
    )
    suffix = "\nOutput:"

    instruction_tokens = tokenizer(instruction, return_tensors="pt")["input_ids"].shape[1]
    suffix_tokens      = tokenizer(suffix,      return_tensors="pt")["input_ids"].shape[1]
    essay_budget       = 512 - instruction_tokens - suffix_tokens - 5

    essay_ids   = tokenizer(row["full_text"], max_length=essay_budget,
                            truncation=True, add_special_tokens=False)["input_ids"]
    essay_trunc = tokenizer.decode(essay_ids, skip_special_tokens=True)
    return instruction + essay_trunc + suffix

## predict_phi
def predict_phi2(test, model, tokenizer, max_input_tokens=512):
    y_pred, y_generated = [], []
    model.eval()

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=max_input_tokens, padding=False,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        generated  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        result = {1: "Pass", 0: "Fail", -1: "???"}[y_pred[-1]]
        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {result}")

    return y_pred, y_generated

In [ ]:
# Evaluate
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

## Llama 3.1 8B Instruct

# Part one

In [ ]:
# Part 1 Llama-3.1-8B-Instruct, 4bit, zero-shot
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, os, gc

# auth (gated model license must be accepted on the HF page)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("B-Llama-3.1-8B-Access")   # hf secret's exact name
from huggingface_hub import whoami
print("logged in as:", whoami()["name"])            # verify BEFORE loading

model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # A100: bf16 compute because we have A100 becareful with this
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id,
      "| eos:", tokenizer.eos_token, tokenizer.eos_token_id)

# Llama 3.1 has NO dedicated pad so we can use eos similar eos to TinyLlama/Mistral situation
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"              # inference first

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id
print("Memory (MB):", round(model.get_memory_footprint() / 1e6, 1))

In [ ]:
# zero-shot: test, then full test-765
one_samsple = X_val_prompts.iloc[:1].reset_index(drop=True)
_, g = predict_decoder_only(one_samsple, model, tokenizer)
print(repr(g[0]))                            # eyeball the raw output first

y_pred, y_generated = predict_decoder_only(X_test_prompts.reset_index(drop=True), model, tokenizer)

pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "generated": y_generated}) \
  .to_csv("/content/drive/MyDrive/llama31_zeroshot_test_results.csv", index=False)

valid = [i for i, p in enumerate(y_pred) if p != -1]
yt = y_true[valid]; yp = [y_pred[i] for i in valid]
print(f"Parsed: {len(valid)}/{len(y_pred)}")
print(f"Accuracy: {accuracy_score(yt, yp):.4f}")
print(classification_report(yt, yp, target_names=["Fail","Pass"]))
print("Predicted-Pass fraction:", (pd.Series(yp)==1).mean(), " (true rate 0.42)")

First time i see this ACC of .69 no finetuned O_O

In [ ]:
#Part 2 lora
# Part 2: QLoRA finetune
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Llama architecture target modules
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)


tokenizer.padding_side = "right"             # training mode...remember why
train_hf = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in train_balanced.iterrows()])
val_hf   = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in val_df.iterrows()])
print(repr(train_hf[0]["prompt"][-90:]))     # want ...<|start_header_id|>assistant<|end_header_id|>...
print(repr(train_hf[0]["completion"]))       # 'Pass' or 'Fail'

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()           # expect ~0.4-0.6%, NOT zero

In [ ]:
# A100 GPU, that it we rich today
# train the stabilized recipe (learned the hard way on Qwen)
sft_config = SFTConfig(
    output_dir="./llama31-lora-balanced",
    num_train_epochs=1,
    per_device_train_batch_size=4,           # 8B is heavier than 7B so lets start conservative
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,           # effective batch = 16
    warmup_steps=100,
    learning_rate=5e-5,
    max_grad_norm=0.3,
    fp16=False, bf16=True,                   # A100 GPU we
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    report_to="none",
    max_length=2000,                                  #i forgot to check llama context windown so im jusst putting 2000....
    completion_only_loss=True,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf.select(range(250)),
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
ADAPTER_PATH = "./llama31-lora-balanced-adapter"

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Saved locally: {ADAPTER_PATH}")
print(f"Files: {os.listdir(ADAPTER_PATH)}")

In [ ]:
import shutil, os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

LOCAL_ADAPTER_PATH = "./llama31-lora-balanced-adapter"
DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/llama31-lora-balanced-adapter"

assert os.path.exists(LOCAL_ADAPTER_PATH), "No local adapter — did the save cell run?"

if os.path.exists(DRIVE_ADAPTER_PATH):
    shutil.rmtree(DRIVE_ADAPTER_PATH)
    print("Removed old version from Drive")

shutil.copytree(LOCAL_ADAPTER_PATH, DRIVE_ADAPTER_PATH)
print(f"   Backed up to Drive: {DRIVE_ADAPTER_PATH}")
print(f"   Files: {os.listdir(DRIVE_ADAPTER_PATH)}")

In [ ]:
tokenizer.padding_side = "left"          # flip back from training's "right"
model_ft = trainer.model                 # best checkpoint via load_best_model_at_end
model_ft.eval()
model_ft.config.use_cache = False

smoke = X_val_prompts.iloc[:1].reset_index(drop=True)
_, g = predict_decoder_only(smoke, model_ft, tokenizer)
print(repr(g[0]))    # want a clean 'pass' or 'fail'

In [ ]:
y_pred, y_generated = predict_decoder_only(
    X_test_prompts.reset_index(drop=True), model_ft, tokenizer
)

# results to Drive!!!!!!!!!!!!!!!!!!!!!!!!
pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "generated": y_generated}) \
  .to_csv("/content/drive/MyDrive/llama31_finetuned_test_results.csv", index=False)

valid = [i for i, p in enumerate(y_pred) if p != -1]
yt = y_true[valid]; yp = [y_pred[i] for i in valid]

print(f"Parsed: {len(valid)}/{len(y_pred)}")
print(f"Accuracy: {accuracy_score(yt, yp):.4f}")
print(classification_report(yt, yp, target_names=["Fail", "Pass"]))
print(confusion_matrix(yt, yp))
print("Predicted-Pass fraction:", (pd.Series(yp)==1).mean(), " (true rate 0.42)")

.89 O_O

In [ ]:
from google.colab import runtime
print("Everything saved. Releasing the runtime.")
runtime.unassign()
# stop the runtime

## HuggingGaceH4 4bit

In [ ]:
# Zephyr 7B with 4bit quantization (required for T4 16GB)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

In [ ]:
#not doing this anymore

# Zephyr uses a ChatML-style template
def generate_zephyr_prompt(row):
    system_msg = "You are a strict essay grader. Respond with one word only: Pass or Fail."
    user_msg = (
        f"Prompt name: {row['prompt_name']}\n"
        f"Task type: {row['task']}\n\n"
        f"Classify the essay as Pass or Fail.\n"
        f"Pass = proficient or above. Fail = developing or below.\n"
        f"Respond with one word only: Pass or Fail.\n\n"
        f"Essay: {row['full_text']}"
    )
    # Zephyr ChatML format
    return (
        f"<|system|>\n{system_msg}</s>\n"
        f"<|user|>\n{user_msg}</s>\n"
        f"<|assistant|>\n"
    )

X_test_prompts_zephyr = pd.DataFrame(
    test_df.apply(generate_zephyr_prompt, axis=1), columns=["text"]
)
print(f"Test prompts: {len(X_test_prompts_zephyr)}")
print("\nExample (first 300 chars):")
print(X_test_prompts_zephyr["text"].iloc[0][:300])

In [ ]:
# Predict function for Zephyr
def predict_zephyr(test, model, tokenizer, max_input_tokens=3500):
    y_pred      = []
    y_generated = []

    model.eval()
    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_tokens,
            padding=False,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,        # greedy — deterministic
                pad_token_id=tokenizer.eos_token_id,
            )

        # decode only newly generated tokens
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        generated  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        result = {1: "Pass", 0: "Fail", -1: "???"}[y_pred[-1]]
        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {result}")

    return y_pred, y_generated


# run on full test set
#y_pred, y_generated = predict_zephyr(X_test_prompts_zephyr, model, tokenizer)

In [ ]:
# run on full test set
y_pred, y_generated = predict_zephyr(X_test_prompts_zephyr, model, tokenizer)

In [ ]:
# Evaluate
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

## Qwen/Qwen2 7B 4bit

PADDING (tokenizer)

What is padding
I guess it is about filling extra space like placeholders tokens because we need that every sequence in a batch ends up with the same length
 * this is for processing GPU process barches as rectangular grids of numbers can not handle rows with different lengths in a single batch
GPU needs every sequence of token lenght to be the same lenght. But lets say essays are of different lenghts.
 * Essay A: "I think cars ar bad."
    * 6 tokens

 * Essay B: "Online school is great."
    * 5 tokens

To Batch, the model will need to pad to length 6
(so we need to padding, this is what i understands)
- Essay A: [i] [think] [cars] [are] [bad] [.]

- Essay B: [online] [school] [is] [great] [.] [PAD]

So now that these are all the same length, they can be stacked into one tensor the GPU can process in parallel. Also, i need to remember that pad has no meeaning when we mask i looks something lik
- Essay 1 input: [I] [like] [cars] [.] [PAD] [PAD]

- Essay 1 input: 1 1 1 1 0 0

(The masking allow the model to ignore the padded positions when computing attention and loss)

In [ ]:
!pip install -U peft trl bitsandbytes accelerate transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('B-Llama-3.1-8B-Access')   # stored securely in Colab secrets
login(token=HF_TOKEN)

What is Padding?
* When we need to process multiple sequences at once (that is a batch like lets say multiple essay)
 * Differetn essays tokenize to differetn lenghts but tensor calculations are must be rectangular so the shorter one (essay) get filled with pad tokens up to the longest one.
 * But Padding asks wehre do we put the filler on? Like which side?
 * NOW for decoder only model
   * remember decoder only model, the whole output extraction section, like a decoder contintues the seque3nce from its end
    * Generation appends new tokens after the last position (the last position of each row is very important.

**So Right Padding**

```
Essay A: [tok tok tok ... tok PAD PAD PAD PAD]
# real content ends early
Essay B: [tok tok tok ... tok tok tok tok tok]  
#full length
```

* If essay A's sequence ends in padding, when the model generates "the next token after the end," it's continuing from a pile of filler and we get -1

**So Left Padding**
* it puts the fillers at the front

```
Essay A: [PAD PAD PAD PAD tok tok tok ... tok]
#real content flush against the END
Essay B: [tok tok tok tok tok tok tok ... tok]
```
But remember
* Now every row ends with its real prompt, and the assistant marker sits at the final position for every essay in the batch
  * and now generation for each one begins exactly where it should.


SO KEEP this in mind, Why training uses right padding?
* padding_side = "right"
  * training doesnt happen from the en, it is computing the whole system and expect real content from the start position 0
   * training padding to the right
   * generation padding to the left

forgetting this would produce the garbage-continuation bug


In [ ]:

# Loading Qwen2 7B with 4-bit quantization (required for T4 16GB)
# I need to chck for  L4 GPU
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2-7B-Instruct"

#Quantization code
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

## toknizer and PADDING

#convert text into numbers, tokenIDs that the model understands
#every model have their own matching tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

#this is about padding [PAD]
#each model has a dedicated PAD only a <EOS> end of sequence token
# tokenizer.pad_token = tokenizer.eos_token see below why not


# where the padding goes
# to the left if for inferences
tokenizer.padding_side = "left"


#auto.. to call a causal language model a model that predicts th next token
#from_pretrained, downloads the pretrained weights for th model
model = AutoModelForCausalLM.from_pretrained( # remember this is from huggingface to call the model
    model_name,                               # the name of the model
    quantization_config=bnb_config,
    device_map="auto",                        # auto tells hf to decide on the avaliable resources: in our case: change "cpu" to "auto" so T4 GPU is used
    token=HF_TOKEN,
)

# model and tokenizer aggreement
#model.config.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded on:", next(model.parameters()).device)
print("Memory (MB)    :", round(model.get_memory_footprint() / 1e6, 1))

Qwen has a  dedicated pad token so
* lets not overwrite it
  * tokenizer.pad_token = tokenizer.eos_token
    * so we not doing this

In [ ]:
#check the pad and oes
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id, "| eos:", tokenizer.eos_token, tokenizer.eos_token_id)

In [ ]:
X = sample.drop(columns=["holistic_essay_score", "binary_score"])
y = sample["binary_score"].copy()          # binary 0/1 instead of 1 to 6, th 1 to 6 was not working at all

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train):,}  ({len(X_train)/len(sample)*100:.1f}%)")
print(f"Val   : {len(X_val):,}   ({len(X_val)/len(sample)*100:.1f}%)")
print(f"Test  : {len(X_test):,}   ({len(X_test)/len(sample)*100:.1f}%)")
print("\nClass distribution (stratification check):")
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = y_split.value_counts().rename({0: "Fail", 1: "Pass"})
    print(f"  {name}: {counts.to_dict()}")

In [ ]:
#Part 2
#better version to keep things consistent
def generate_test_prompt(row):
    return (
        f"You are an expert essay grader. Read the essay and respond with exactly one word.\n"
        f"Your response must be either the word Fail or Pass. No other words.\n\n"
        f"Rules:\n"
        f"- Fail: the essay is weak, underdeveloped, or below standard\n"
        f"- Pass: the essay is proficient, strong, or excellent\n\n"
        f"Prompt: {row['prompt_name']}\n"
        f"Task: {row['task']}\n"
        f"Essay: {row['full_text']}\n\n"
        f"Respond with one word only (Fail or Pass): "
    )


def make_prompt_completion(row, tokenizer):
    user_content = generate_test_prompt(row)   # same wording
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False, add_generation_prompt=True,
    )
    completion = "Pass" if row["binary_score"] == 1 else "Fail"
    return {"prompt": prompt, "completion": completion}

#Train
train_df = X_train.copy()
train_df["binary_score"] = y_train.values

# eval prompt
val_df = X_val.copy()
val_df["binary_score"] = y_val.values
X_val_prompts = pd.DataFrame(val_df.apply(generate_test_prompt, axis=1), columns=["text"])

test_df = X_test.copy()
test_df["binary_score"] = y_test.values
y_true = test_df["binary_score"].values
X_test_prompts = pd.DataFrame(test_df.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
def predict_decoder_only(test, model, tokenizer):
    y_pred      = []
    y_generated = []

    model.eval()  # disable training only behavior (we not training here)

    # loop through every prompt in the test set
    for i in tqdm(range(len(test))):  # tqdm gives the progress bar, really helpful

        messages = [{"role": "user", "content": test.iloc[i]["text"]}]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        # convert the prompt string into token IDs (numeric)
        # we are giving all of this to our model in a language it can understand (tokenize)
        inputs = tokenizer(
            prompt,             #this is the prompt
            return_tensors="pt",  #tensor
            truncation=True,      # cut, not sure how this affect the evaluation
            max_length=10000,   # Llama handles longer context than T5's 512, anything more 2048 bye bye
            padding=False      # one essay at a time, nothing to pad against, less problems
        ).to(model.device)     # move the tokenized prompt to wherever the model lives (cpu/gpu)

        # remember how long the INPUT was, in tokens
        # decoder-only models return input + answer glued together,
        # so we need this to cut the input back off afterwards

        input_len = inputs["input_ids"].shape[1]

        # no gradient tracking, we are not training
        with torch.no_grad():
            # Llama GPT are decoder only no not forget this so --->>>input and output share one stream,
            # so .generate() APPENDS the answer onto the prompt tokens
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,         # only need a word or two ("Pass"/"Fail")
                do_sample=False,           # greedy = deterministic, reproducible in a workshop
                #pad_token_id=tokenizer.eos_token_id,  #this has to be changed
                pad_token_id=tokenizer.pad_token_id,

                use_cache=False,
            )
        # Where the answerrrrr
        # The key step here: keep only the tokens AFTER the prompt.
        # outputs[0] = [ ...prompt tokens... , ...new answer tokens... ]
        # so lets slice from input_len onward to get ONLY what the model added.

        new_tokens = outputs[0][input_len:]

        # turn those new token IDs back into text
        # skip_special_tokens drops <|assistant|>, </s>, etc.
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

        y_generated.append(generated)

        # map the text answer to a label, So
        # contains "pass" -> 1, contains "fail" -> 0, neither -> -1 (unparseable problems)
        if "pass" in generated: y_pred.append(1)
        elif "fail" in generated: y_pred.append(0)
        else: y_pred.append(-1)

        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  → → {y_pred[-1]}")

    return y_pred, y_generated

.generate() is actually a universal method available for every hf model that can produce text
   * I thought it was only for encoder-decoder model
   * Qwen is a decoder only model lik GPT and llama

Also, remember
* Decoder only model: genera text by predicting th next words based on previous words
* Encoder-decoder models are the input sequence and translate it into an entirely different output sequence. translation model english to spanish

In [ ]:
# zero-shot Qwen baseline — same 100 val essays as Phi's 0.58 baseline
val_sample   = X_val_prompts.iloc[:100].reset_index(drop=True)
y_val_sample = y_val.values[:100]

y_pred, y_generated = predict_decoder_only(val_sample, model, tokenizer)

valid = [i for i, p in enumerate(y_pred) if p != -1]
yt = y_val_sample[valid]
yp = [y_pred[i] for i in valid]

print(f"Parsed: {len(valid)}/100")
print(f"Accuracy: {accuracy_score(yt, yp):.4f}   (Phi-3.5 zero-shot was 0.58)")
print(classification_report(yt, yp, target_names=["Fail", "Pass"]))
print(confusion_matrix(yt, yp))
print("Predicted-Pass fraction:", (pd.Series(yp) == 1).mean(), " (true rate 0.42)")

In [ ]:
y_pred, y_generated = predict_decoder_only(X_test_prompts.reset_index(drop=True), model, tokenizer)

In [ ]:
# Evaluate
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

#### LoRA fintuning Qwen2 7B

import gc
import torch

In [ ]:
import gc
import torch

In [ ]:
# check which model variables exist in memory
model_vars = ["model", "ft_model", "base_model", "trainer"]
for var in model_vars:
    if var in dir():
        obj = eval(var)
        try:
            mb = round(obj.get_memory_footprint() / 1e6, 1)
            print(f"  {var:<12} → loaded  ({mb} MB)")
        except:
            print(f"  {var:<12} → loaded  (size unknown)")
    else:
        print(f"  {var:<12} → not in memory")

# overall GPU memory summary
print()
total  = torch.cuda.get_device_properties(0).total_memory / 1e9
free   = torch.cuda.mem_get_info()[0] / 1e9
used   = total - free
print(f"GPU total : {total:.2f} GB")
print(f"GPU used  : {used:.2f} GB")
print(f"GPU free  : {free:.2f} GB")

In [ ]:
#  delete all loaded models
for var in ["model", "ft_model", "base_model", "trainer"]:
    if var in dir():
        exec(f"del {var}")
        print(f"  deleted {var}")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# check result
total = torch.cuda.get_device_properties(0).total_memory / 1e9
free  = torch.cuda.mem_get_info()[0] / 1e9
used  = total - free
print(f"\nGPU total : {total:.2f} GB")
print(f"GPU used  : {used:.2f} GB")
print(f"GPU free  : {free:.2f} GB")

In [ ]:
import torch.nn as nn
print({n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)})
# expect: {'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj','lm_head'}

In [ ]:
# Cell 1: Imports & config
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch
import gc

MODEL_NAME   = "Qwen/Qwen2-7B-Instruct"
ADAPTER_PATH = "./qwen2-lora-balanced-adapter"

# 4-bit quantization — float16 required for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[            # Qwen2 attention + MLP layers
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
#Part ??????????

In [ ]:
train_df = X_train.copy();  train_df["binary_score"] = y_train.values
val_df   = X_val.copy();    val_df["binary_score"]   = y_val.values
X_val_prompts  = pd.DataFrame(val_df.apply(generate_test_prompt, axis=1), columns=["text"])
test_df  = X_test.copy();   test_df["binary_score"]  = y_test.values
y_true   = test_df["binary_score"].values
X_test_prompts = pd.DataFrame(test_df.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
fail_df = train_df[train_df["binary_score"] == 0]
pass_df = train_df[train_df["binary_score"] == 1]
train_balanced = pd.concat([
    fail_df.sample(n=len(pass_df), random_state=42),
    pass_df,
]).sample(frac=1, random_state=42).reset_index(drop=True)
print(train_balanced["binary_score"].value_counts().to_dict())

In [ ]:
tokenizer.padding_side = "right"    # right for training (back to left for inference)

train_hf = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in train_balanced.iterrows()])
val_hf   = Dataset.from_list([make_prompt_completion(r, tokenizer) for _, r in val_df.iterrows()])

print(f"Train : {len(train_hf)}  Val : {len(val_hf)}")
print("Prompt ending:", repr(train_hf[0]["prompt"][-70:]))   # want ...<|im_end|>\n<|im_start|>assistant\n
print("Completion   :", repr(train_hf[0]["completion"]))     # 'Pass' or 'Fail'

In [ ]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Notes

* increase max_length=3000, increases the timing like 1h more

In [ ]:
lens = [len(tokenizer.encode(ex["prompt"] + ex["completion"])) for ex in train_hf]
print("max:", max(lens), "| over 1024:", sum(l > 1024 for l in lens), "| over 3000:", sum(l > 3000 for l in lens))

In [ ]:
#A100 GPU, 20m
sft_config = SFTConfig(
    output_dir="./qwen2-lora-balanced",
    num_train_epochs=1,                      # verify this is what actually runs!
    per_device_train_batch_size=8,           # A100: real batches
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,           # effective batch = 16
    warmup_steps=100,                        # stability
    learning_rate=5e-5,
    max_grad_norm=0.3,
    fp16=False, bf16=True,                   # A100
    logging_steps=10,                        # more frequent logs on a shorter run
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    report_to="none",
    max_length=2000,                         # need to check qwen for sure its content window is not 2000
    completion_only_loss=True,
    optim="paged_adamw_8bit",
)



trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf.select(range(250)),
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
# inference setup + smoke test (defines model_ft)
tokenizer.padding_side = "left"
model_ft = trainer.model
model_ft.eval()
model_ft.config.use_cache = False

smoke = X_val_prompts.iloc[:1].reset_index(drop=True)
_, g = predict_decoder_only(smoke, model_ft, tokenizer)
print(repr(g[0]))    # want a clean 'pass' or 'fail'

In [ ]:
y_pred, y_generated = predict_decoder_only(X_test_prompts.reset_index(drop=True), model_ft, tokenizer)

pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "generated": y_generated}) \
  .to_csv("/content/drive/MyDrive/qwen2_finetuned_test_results.csv", index=False)   # straight to Drive

valid = [i for i, p in enumerate(y_pred) if p != -1]
yt = y_true[valid]; yp = [y_pred[i] for i in valid]
print(f"Parsed: {len(valid)}/{len(y_pred)}")
print(f"Accuracy: {accuracy_score(yt, yp):.4f}")
print(classification_report(yt, yp, target_names=["Fail", "Pass"]))
print(confusion_matrix(yt, yp))

In [ ]:
#from downloading it from drive
# ── 1. get the adapter from Drive (Cell B pattern) ──
from google.colab import drive
import shutil, os
drive.mount("/content/drive")

LOCAL  = "./qwen2-lora-balanced-adapter"
DRIVE  = "/content/drive/MyDrive/qwen2-lora-balanced-adapter"

if not os.path.exists(LOCAL):
    shutil.copytree(DRIVE, LOCAL)
print("Files:", os.listdir(LOCAL))   # expect adapter_model.safetensors, adapter_config.json, tokenizer files

# ── 2. load base model (4-bit) + attach the trained adapter ──
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

model_ft = PeftModel.from_pretrained(base, LOCAL)
model_ft.eval()
model_ft.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(LOCAL)   # the saved tokenizer, with chat template
tokenizer.padding_side = "left"                    # inference


In [ ]:
#3. use it — same predict function as always ──
y_pred, y_generated = predict_decoder_only(X_test_prompts, model_ft, tokenizer)

In [ ]:
from google.colab import runtime
print("Adapter safe on Drive. Releasing the A100.")
runtime.unassign()

### testing

In [ ]:
# L4 GPU 4h crashing not working T_T, 3 times!!
sft_config = SFTConfig(
    output_dir="./qwen2-lora-balanced",     # Qwen path
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=False, bf16=False,
    logging_steps=25,
    eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=100,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    report_to="none",
    max_length=2000,
    completion_only_loss=True,
    optim="paged_adamw_8bit",
)

# fp32 cast, same cell as trainer
for n, p in model.named_parameters():
    if p.requires_grad: p.data = p.data.float()

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf.select(range(250)),
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
# Save adapter locally
ADAPTER_PATH = "./qwen2-lora-balanced-adapter"

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Saved locally: {ADAPTER_PATH}")
print(f"Files: {os.listdir(ADAPTER_PATH)}")

In [ ]:
# Backup to Drive
import shutil, os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

LOCAL_ADAPTER_PATH = "./qwen2-lora-balanced-adapter"
DRIVE_ADAPTER_PATH = "/content/drive/MyDrive/qwen2-lora-balanced-adapter"

# safety: never touch Drive unless the fresh local adapter exists
assert os.path.exists(LOCAL_ADAPTER_PATH), "No local adapter — did the save cell run?"

if os.path.exists(DRIVE_ADAPTER_PATH):
    shutil.rmtree(DRIVE_ADAPTER_PATH)
    print("Removed old version from Drive")

shutil.copytree(LOCAL_ADAPTER_PATH, DRIVE_ADAPTER_PATH)
print(f"   Backed up to Drive: {DRIVE_ADAPTER_PATH}")
print(f"   Files: {os.listdir(DRIVE_ADAPTER_PATH)}")

In [ ]:
# Kill the runtime after everything is saved
from google.colab import runtime
print("All saved. Terminating runtime good night.")
runtime.unassign()

In [ ]:
# Cell ??: Load base model and apply LoRA
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.eos_token_id
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)

# cast trainable LoRA layers to float16 (avoids bfloat16 error on T4)
for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

dtypes = set(p.dtype for p in model.parameters())
print("Dtypes in model:", dtypes)    # should NOT contain torch.bfloat16 im getting error T_T
model.print_trainable_parameters()

In [ ]:
# Fine-tune T4 GPU
# fp16/bf16 disabled avoids  bfloat16 error on T4
# Qwen2 has 128k context but we cap at 2048 to fit T4 memory
sft_config = SFTConfig(
    output_dir="./qwen2-lora-balanced",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      # effective batch size = 8
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    max_length=2048,                    # Qwen2 supports 128k but T4 caps here
    dataset_text_field="text",
    loss_type="nll",
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()

In [ ]:
# clean up a corrupted state
# Step 1 clean up corrupted state
for var in ["model", "ft_model", "base_model", "trainer"]:
    if var in dir():
        exec(f"del {var}")
        print(f"  deleted {var}")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("Free GPU (GB):", round(torch.cuda.mem_get_info()[0] / 1e9, 2))

In [ ]:
#Save adapter
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")

In [ ]:
#Load fine-tuned model for inference
gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "left"     # switch to left padding for inference

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()

X_test_prompts_qwen_ft = pd.DataFrame(
    test_df.apply(generate_qwen_test_prompt, axis=1), columns=["text"]
)
print(f"Test prompts : {len(X_test_prompts_qwen_ft)}")
print("Last 50 chars:", repr(X_test_prompts_qwen_ft["text"].iloc[0][-50:]))
# should end with '<|im_start|>assistant\n'

In [ ]:
# Predict
def predict_qwen_ft(test, model, tokenizer, max_input_tokens=2048):
    y_pred, y_generated = [], []
    model.eval()

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=max_input_tokens, padding=False,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=3,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        generated  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        result = {1: "Pass", 0: "Fail", -1: "???"}[y_pred[-1]]
        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {result}")

    return y_pred, y_generated


y_pred, y_generated = predict_qwen_ft(X_test_prompts_qwen_ft, ft_model, tokenizer)

In [ ]:
# Cell 10 Evaluate
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

0.85 O_O

In [ ]:
# save
from google.colab import drive
import shutil
drive.mount("/content/drive")
shutil.copytree(ADAPTER_PATH, f"/content/drive/MyDrive/{ADAPTER_PATH}")
print("Backed up to Drive")

In [ ]:
#
from google.colab import drive
import shutil
drive.mount("/content/drive")
shutil.copytree(ADAPTER_PATH, f"/content/drive/MyDrive/{ADAPTER_PATH}")
print("Backed up to Drive")

##### downloading and testing adapter

In [ ]:

from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

local_path = "./qwen2-lora-balanced"                          # where trainer saved checkpoints
drive_path = "/content/drive/MyDrive/qwen2-lora-balanced-adapter"

# find the best checkpoint folder saved by trainer
checkpoints = [f for f in os.listdir(local_path) if f.startswith("checkpoint")]
print("Checkpoints found:", checkpoints)

In [ ]:
#prediction

In [ ]:
def predict(test, model, tokenizer):
    """Zero-shot binary prediction. Returns (y_pred, y_generated).
    Returns -1 for unparseable outputs.
    """
    y_pred = []
    y_generated = []
    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=5,
        do_sample=False,
    )
    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        result = pipe(prompt)
        generated = result[0]["generated_text"][len(prompt):].strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)          # unparseable

    return y_pred, y_generated

In [ ]:
def predict(test, model, tokenizer):
    y_pred = []
    y_generated = []

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=5,
        temperature=0.1,
        do_sample=True,
    )

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        result = pipe(prompt)

        # grab only the newly generated text after the prompt
        generated = result[0]["generated_text"][len(prompt):].strip()
        y_generated.append(generated)

        # parse Pass / Fail (case-insensitive)
        lower = generated.lower()
        if "pass" in lower:
            y_pred.append(1)
        elif "fail" in lower:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        # print each response as it comes in
        label = {1: "Pass", 0: "Fail", -1: "???"}[y_pred[-1]]
        print(f"[{i+1:>3}/{len(test)}] Generated: {repr(generated):<20}  →  {label}")

    return y_pred, y_generated

In [ ]:
def predict(test, model, tokenizer):
    y_pred = []
    y_generated = []

    # get the single token id for "Pass" and "Fail"
    pass_id = tokenizer.encode("Pass", add_special_tokens=False)[0]
    fail_id = tokenizer.encode("Fail", add_special_tokens=False)[0]
    print(f"Token ids  —  Pass: {pass_id}, Fail: {fail_id}")

    model.eval()
    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        # logits for the very next token the model would generate
        next_token_logits = outputs.logits[0, -1, :]

        pass_logit = next_token_logits[pass_id].item()
        fail_logit = next_token_logits[fail_id].item()

        label = "Pass" if pass_logit > fail_logit else "Fail"
        y_pred.append(1 if label == "Pass" else 0)
        y_generated.append(f"Pass={pass_logit:.2f} Fail={fail_logit:.2f}")

        print(f"[{i+1:>3}/{len(test)}]  Pass logit: {pass_logit:6.2f}  "
              f"Fail logit: {fail_logit:6.2f}  →  {label}")

    return y_pred, y_generated

In [ ]:
### save the models

In [ ]:
import os

adapter_local = "./phi2-lora-balanced-adapter"
if os.path.exists(adapter_local):
    files = os.listdir(adapter_local)
    print(f"Found adapter at {adapter_local}")
    print(f"Files: {files}")
else:
    print("Adapter NOT found on local disk — may have been lost if runtime restarted")

In [ ]:
# Step 3: copy to Google Drive
import shutil

drive_path = "/content/drive/MyDrive/phi2-lora-balanced-adapter"

shutil.copytree(adapter_local, drive_path)
print(f"Adapter saved to Google Drive at:\n  {drive_path}")

# Ollama

In [ ]:
!nvidia-smi

In [ ]:
!apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
#we need to start the serve

In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [ ]:
!ollama ps

In [ ]:
!ollama pull llama3.2

In [ ]:
!ollama run llama3.2 "hi"

Local host
  * is where the ollama seerver is listening for request. localhost 11434
    * end this request to the Ollama server running right here on this same Colab machine, knocking on door 11434.

In [ ]:
import requests
r = requests.post("http://localhost:11434/api/generate", json={
    "model": "llama3.2:latest",
    "prompt": "Say hello in one sentence",
    "stream": False
})
print(r.json()["response"])

## Llama3.2:latest

In [ ]:
def generate_llama_prompt(row):
    return (
        f"You are a fair essay grader. Read the essay below carefully.\n"
        f"Prompt name: {row['prompt_name']}\n"
        f"Task type: {row['task']}\n\n"
        f"Classify the essay as Pass or Fail.\n"
        f"Pass = the essay meets an acceptable standard (proficient or above).\n"
        f"Fail = the essay does not meet the standard (developing or below).\n"
        f"Respond with one word only: Pass or Fail.\n\n"
        f"Essay: {row['full_text']}"
    )

X_test_prompts_llama = pd.DataFrame(
    test_df.apply(generate_llama_prompt, axis=1), columns=["text"]
)

In [ ]:
import requests
from tqdm import tqdm

def predict_ollama(test, model_name="llama3.2:latest"):
    y_pred, y_generated = [], []

    for i in tqdm(range(len(test))):
        prompt = test.iloc[i]["text"]

        resp = requests.post("http://localhost:11434/api/chat", json={
            "model": model_name,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": {
                "temperature": 0,      # greedy / deterministic, like do_sample=False
                "num_predict": 5,      # cap new tokens, like max_new_tokens=5
                "num_ctx": 4096,       # context window
            },
            "keep_alive": -1,          # keep model pinned on GPU between calls
        })

        generated = resp.json()["message"]["content"].strip().lower()
        y_generated.append(generated)

        if "pass" in generated:
            y_pred.append(1)
        elif "fail" in generated:
            y_pred.append(0)
        else:
            y_pred.append(-1)

        result = {1: "Pass", 0: "Fail", -1: "???"}[y_pred[-1]]
        print(f"[{i+1:>3}/{len(test)}]  raw='{generated}'  →  {result}")

    return y_pred, y_generated


y_pred, y_generated = predict_ollama(X_test_prompts_llama)

In [ ]:
# Evaluate
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

0.76 from ollama models very good, expect no less from ollama

## Qwen

In [ ]:
## Qwen2.5:7b

In [ ]:
!ollama pull qwen2.5:7b

In [ ]:
y_pred, y_generated = predict_ollama(X_test_prompts_llama, model_name="qwen2.5:7b")

In [ ]:
results_df = pd.DataFrame({
    "y_true"    : y_true,
    "y_pred"    : y_pred,
    "generated" : y_generated,
})
results_df["y_true_label"] = results_df["y_true"].map({1: "Pass", 0: "Fail"})
results_df["y_pred_label"] = results_df["y_pred"].map({1: "Pass", 0: "Fail", -1: "???"})
print(results_df.to_string())

valid_mask   = [i for i, p in enumerate(y_pred) if p != -1]
y_true_valid = y_true[valid_mask]
y_pred_valid = [y_pred[i] for i in valid_mask]

print(f"\nParsed      : {len(valid_mask)}/{len(y_pred)}")
print(f"Unparseable : {len(y_pred) - len(valid_mask)}")

if y_pred_valid:
    print(f"\nAccuracy : {accuracy_score(y_true_valid, y_pred_valid):.4f}")
    print(classification_report(
        y_true_valid, y_pred_valid,
        labels=[0, 1], target_names=["Fail", "Pass"], zero_division=0
    ))
    cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           Fail  Pass")
    for label, row in zip(["Fail", "Pass"], cm):
        print(f"True {label:<5}: {row}")

In [ ]:
!ollama list